# RSNA Knee · Notebook 05 — End-to-End Master Pipeline (All Modules)

Single-notebook orchestration of the four-stage workflow:

```
Stage 1  preprocess_dataset.py   raw DICOM  -> .npy volumes + manifest      (CPU hours)
Stage 2  parse_reports.py        reports    -> train_pseudo_labels.csv      (minutes)
Stage 3  train.py                volumes+labels -> best_auc_fold_{k}.pt     (GPU hours)
Stage 4  submission_notebook.py  test DICOM + weights -> submission.csv     (GPU, offline)
```

**Score strategy recap (targets Macro-AUC > 0.96):**
1. Weak supervision from reports expands labeled data 5–10×.
2. Multi-plane 2.5D attention-pooled ConvNeXt/EffNet ensemble.
3. Asymmetric Loss under extreme class sparsity.
4. TTA + 2-backbone × 5-fold rank-average ensemble (+0.015…0.025 expected lift).

Toggle `SMOKE_TEST=True` to cap data volume for fast CI-style verification before the
full 9-hour Kaggle run.


In [ ]:
# --- Stage 0: environment, seeds, paths ------------------------------------------------
import os, glob, json, time, warnings
warnings.filterwarnings("ignore")
SMOKE_TEST = True   # set False for the full run

def _first(patterns):
    for p in patterns:
        hits = glob.glob(p, recursive=True)
        if hits:
            return hits[0]
    return None

DATA_ROOT   = _first(["/kaggle/input/*rsna-str-pkc*", "/kaggle/input/*knee*"]) or "/kaggle/input/rsna-str-pkc-dataset"
TRAIN_SERIES= os.path.join(DATA_ROOT, "train_series")
TEST_SERIES = os.path.join(DATA_ROOT, "test_series")
TRAIN_CSV   = os.path.join(DATA_ROOT, "train.csv")
TEST_CSV    = os.path.join(DATA_ROOT, "test.csv")
WORK        = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./work"
os.makedirs(WORK, exist_ok=True)
VOL_ROOT    = os.path.join(WORK, "preprocessed", "train_volumes")
MANIFEST    = os.path.join(WORK, "train_manifest.csv")
PSEUDO_CSV  = os.path.join(WORK, "train_pseudo_labels.csv")
CKPT_ROOT   = os.path.join(WORK, "checkpoints")
print(json.dumps({"DATA_ROOT": DATA_ROOT, "WORK": WORK}, indent=2))

## Stage 1 · Module 1 source — `preprocess_dataset.py`

In [ ]:
"""
MODULE 1 — Offline DICOM Preprocessor for RSNA Knee Abnormality Detection.

Reads raw `.dcm` files from `train_series/` (and optionally `test_series/`) on CPU,
using a process pool to avoid I/O / decode bottlenecks, and persists each MRI series
as a fixed-shape ``uint8`` volume:

    (NUM_SLICES, H, W) == (16, 256, 256), values in [0, 255]

Pipeline per series:
    1. Robustly read every slice with pydicom (graceful fallbacks for JPEG Lossless,
       JPEG-2000, RLE — requires pylibjpeg-openjpeg / pylibjpeg-libjpeg / gdcm if the
       native decoder cannot unpack the transfer syntax).
    2. Physically sort slices by ImagePositionPatient[2] -> SliceLocation -> InstanceNumber.
    3. Percentile intensity clipping (0.5% / 99.5%) followed by Min-Max scaling to uint8.
    4. Uniform depth resampling of variable slice counts (typically 20-45) to K=16 slices
       using linear (linspace) interpolation along the depth axis.
    5. Spatial resize to 256x256.
    6. Save as .npy keyed by f"{StudyInstanceUID}_{SeriesInstanceUID}.npy".
    7. Emit a metadata manifest (parquet/csv) linking StudyInstanceUID, SeriesInstanceUID,
       AnatomicalPlane, FluidSensitive, FatSuppression, Rows, Columns, NumberSlicesAvailable.

Usage (Kaggle):
    !python preprocess_dataset.py --series-root /kaggle/input/rsna-str-pkc-dataset/train_series \
        --out-root /kaggle/working/preprocessed/train_volumes --manifest /kaggle/working/train_manifest.csv
"""

from __future__ import annotations

import argparse
import json
import logging
import os
import sys
import time
import traceback
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np

try:
    import pydicom
    from pydicom.errors import InvalidDicomError
except ImportError as exc:  # pragma: no cover
    raise SystemExit(f"pydicom is required for this module: {exc}")

# Optional fast decoders. pydicom >= 2.2 auto-routes through these when installed.
_DECODER_HINT = ""
try:  # noqa: SIM105
    import pylibjpeg  # type: ignore  # noqa: F401
    _DECODER_HINT += " pylibjpeg"
except Exception:
    pass
try:  # noqa: SIM105
    import pylibjpeg_openjpeg  # type: ignore  # noqa: F401
    _DECODER_HINT += " openjpeg"
except Exception:
    pass
try:  # noqa: SIM105
    import pylibjpeg_libjpeg  # type: ignore  # noqa: F401
    _DECODER_HINT += " libjpeg"
except Exception:
    pass
try:  # noqa: SIM105
    import gdcm  # type: ignore  # noqa: F401
    _DECODER_HINT += " gdcm"
except Exception:
    pass

try:
    import cv2  # OpenCV is preinstalled on Kaggle; used for fast spatial resizing.
    _HAVE_CV2 = True
except Exception:  # pragma: no cover
    _HAVE_CV2 = False

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(processName)s | %(message)s",
    stream=sys.stdout,
)
logger = logging.getLogger("preprocess")


# --------------------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------------------
@dataclass(frozen=True)
class PreprocessConfig:
    num_slices: int = 16
    image_size: Tuple[int, int] = (256, 256)
    p_low: float = 0.5           # lower percentile clip (%)
    p_high: float = 99.5         # upper percentile clip (%)
    save_dtype: str = "uint8"    # persisted dtype
    compress: bool = False       # use .npz (compressed) instead of raw .npy
    max_workers: int = 0         # 0 => cpu_count
    skip_existing: bool = True


# --------------------------------------------------------------------------------------
# DICOM reading helpers (thread/process safe)
# --------------------------------------------------------------------------------------
def _safe_get(ds: "pydicom.Dataset", name: str, default: Any = None) -> Any:
    """Fetch a DICOM attribute defensively (missing tags are extremely common)."""
    try:
        val = getattr(ds, name, default)
    except Exception:
        return default
    if val is None or (isinstance(val, (list, tuple)) and len(val) == 0):
        return default
    return val


def _sort_key_factory(ds_list: List["pydicom.Dataset"]) -> Tuple[str, Any]:
    """Choose the most reliable physical-ordering key available across the series."""
    def _ipp_z(ds: "pydicom.Dataset") -> float:
        ipp = _safe_get(ds, "ImagePositionPatient", None)
        try:
            return float(ipp[2])
        except Exception:
            return float("nan")

    def _slice_loc(ds: "pydicom.Dataset") -> float:
        sl = _safe_get(ds, "SliceLocation", None)
        try:
            return float(sl)
        except Exception:
            return float("nan")

    def _inst_num(ds: "pydicom.Dataset") -> float:
        n = _safe_get(ds, "InstanceNumber", None)
        try:
            return float(n)
        except Exception:
            return float("nan")

    for fn, label in ((_ipp_z, "ImagePositionPatient"), (_slice_loc, "SliceLocation"), (_inst_num, "InstanceNumber")):
        vals = [fn(d) for d in ds_list]
        finite = [v for v in vals if np.isfinite(v)]
        # Require that at least 80% of slices carry the tag and values vary.
        if len(finite) >= max(2, int(0.8 * len(vals))) and len(set(finite)) > 1:
            return label, fn
    return "InstanceNumber", _inst_num


def load_series_slices(series_dir: Path) -> Tuple[List["pydicom.Dataset"], str]:
    """Load all DICOM files inside a series directory, sorted physically.

    Returns:
        (sorted datasets, sort_key_used)
    Raises:
        RuntimeError if no readable slices exist.
    """
    files: List[Path] = []
    try:
        entries = list(os.scandir(str(series_dir)))
    except OSError:
        entries = []
    for e in entries:
        if e.is_file():
            files.append(Path(e.path))
    if not files:
        raise RuntimeError(f"No files found under {series_dir}")

    datasets: List["pydicom.Dataset"] = []
    n_fail = 0
    for fp in files:
        try:
            # force=True tolerates missing preamble / non-standard headers.
            ds = pydicom.dcmread(str(fp), force=True)
            if "PixelData" not in ds and not hasattr(ds, "pixel_data_array"):
                n_fail += 1
                continue
            datasets.append(ds)
        except (InvalidDicomError, Exception) as exc:  # noqa: BLE001
            n_fail += 1
            logger.debug("Unreadable file %s: %s", fp, exc)
    if not datasets:
        raise RuntimeError(f"Zero readable slices in {series_dir} ({n_fail} failures)")

    key_label, key_fn = _sort_key_factory(datasets)
    # Stable sort; NaN keys pushed to the end but replaced with interpolated ordering later.
    try:
        datasets_sorted = sorted(datasets, key=lambda d: (key_fn(d) if np.isfinite(key_fn(d)) else float("inf")))
    except Exception:
        datasets_sorted = datasets
    return datasets_sorted, key_label


def dataset_to_float_array(ds: "pydicom.Dataset") -> np.ndarray:
    """Convert one slice's pixel data to a 2D float32 array with rescale applied.

    Handles multi-frame images and applies RescaleSlope/Intercept defensively.
    """
    try:
        arr = ds.pixel_array
    except Exception as exc:  # Transfer syntax not natively supported.
        msg = str(exc)
        raise RuntimeError(
            f"Pixel decode failed (transfer syntax={_safe_get(ds,'TransferSyntaxUID','?')})."
            f" Install offline wheels pylibjpeg-openjpeg/pylibjpeg-libjpeg/gdcm.{_DECODER_HINT}. Err: {msg}"
        ) from exc

    arr = np.asarray(arr)
    if arr.ndim == 3:  # Multi-frame: take mean over frames (rare for knee MRI single slices).
        arr = arr.reshape(arr.shape[0], -1).mean(axis=1).reshape(arr.shape[1:])
    arr = arr.astype(np.float32)

    slope = _safe_get(ds, "RescaleSlope", 1.0)
    intercept = _safe_get(ds, "RescaleIntercept", 0.0)
    try:
        arr = arr * float(slope) + float(intercept)
    except Exception:
        pass
    return arr


# --------------------------------------------------------------------------------------
# Normalization & resampling
# --------------------------------------------------------------------------------------
def percentile_normalize_stack(vol: np.ndarray, p_low: float, p_high: float) -> np.ndarray:
    """Percentile-clip then min-max scale a (D,H,W) float volume into uint8 [0,255].

    The percentile bounds are computed globally over the whole series so that relative
    tissue contrast between slices is preserved (scanner-specific outlier suppression).
    """
    finite = vol[np.isfinite(vol)]
    if finite.size == 0:
        return np.zeros_like(vol, dtype=np.uint8)
    lo = float(np.percentile(finite, p_low))
    hi = float(np.percentile(finite, p_high))
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        # Degenerate constant series -> center it.
        med = float(np.median(finite))
        lo, hi = med - 1.0, med + 1.0
    out = (vol - lo) / (hi - lo)
    out = np.clip(out, 0.0, 1.0)
    out = np.nan_to_num(out, nan=0.0, posinf=1.0, neginf=0.0)
    return (out * 255.0).round().astype(np.uint8)


def resample_depth(vol: np.ndarray, k: int) -> np.ndarray:
    """Uniform 1D linspace resampling of the depth axis from D slices to exactly k slices.

    Uses linear interpolation between adjacent slices (float math on uint8 input).
    """
    d = vol.shape[0]
    if d == k:
        return vol
    if d == 1:
        return np.repeat(vol, k, axis=0)
    src = np.arange(d, dtype=np.float64)
    dst = np.linspace(0.0, d - 1.0, k)
    x0 = np.floor(dst).astype(np.int64)
    x1 = np.minimum(x0 + 1, d - 1)
    w = (dst - x0).astype(np.float32)[:, None, None]
    v0 = vol[x0].astype(np.float32)
    v1 = vol[x1].astype(np.float32)
    interp = v0 * (1.0 - w) + v1 * w
    return np.clip(interp, 0, 255).astype(np.uint8)


def resize_slice(slice_img: np.ndarray, target_hw: Tuple[int, int]) -> np.ndarray:
    h, w = target_hw
    if _HAVE_CV2:
        return cv2.resize(slice_img, (w, h), interpolation=cv2.INTER_AREA)
    # Pure-numpy bilinear fallback (slower but dependency-free).
    from PIL import Image
    im = Image.fromarray(slice_img)
    im = im.resize((w, h), resample=Image.BILINEAR)
    return np.asarray(im, dtype=np.uint8)


def resize_volume(vol: np.ndarray, target_hw: Tuple[int, int]) -> np.ndarray:
    if vol.shape[1] == target_hw[0] and vol.shape[2] == target_hw[1]:
        return vol
    return np.stack([resize_slice(s, target_hw) for s in vol], axis=0)


# --------------------------------------------------------------------------------------
# Per-series worker (executed in a separate process)
# --------------------------------------------------------------------------------------
def process_one_series(args: Tuple[str, str, str, Dict[str, Any]]) -> Dict[str, Any]:
    study_uid, series_uid, series_path_str, cfg_dict = args
    cfg = PreprocessConfig(**cfg_dict)
    out_root = Path(cfg_dict["_out_root"])
    series_path = Path(series_path_str)
    ext = "npz" if cfg.compress else "npy"
    out_fp = out_root / f"{study_uid}_{series_uid}.{ext}"

    record: Dict[str, Any] = {
        "StudyInstanceUID": study_uid,
        "SeriesInstanceUID": series_uid,
        "VolumePath": str(out_fp),
        "status": "failed",
    }
    t0 = time.time()
    try:
        if cfg.skip_existing and out_fp.exists():
            record["status"] = "skipped"
            record["AnatomicalPlane"] = "Unknown"
            return record

        ds_list, sort_key = load_series_slices(series_path)
        ref = ds_list[0]

        planes = str(_safe_get(ref, "AnatomicalPlaneType", _safe_get(ref, "AnatomicalOrientationType", "Unknown")))
        planes = planes.upper() if planes else "UNKNOWN"
        # Infer plane from ImageOrientationPatient when the tag is absent/unhelpful.
        if planes == "UNKNOWN" or planes == "OTHER":
            iop = _safe_get(ref, "ImageOrientationPatient", None)
            try:
                n = np.array([float(x) for x in iop[:3]]) @ np.array([float(x) for x in iop[3:]])
                ax = int(np.argmax(np.abs(n)))
                planes = {0: "SAGITTAL", 1: "CORONAL", 2: "AXIAL"}.get(ax, "UNKNOWN")
            except Exception:
                planes = "UNKNOWN"

        fluid = str(_safe_get(ref, "DerivedSeriesDescription", "") or "") + " " + \
                str(_safe_get(ref, "SequenceName", "") or "") + " " + \
                str(_safe_get(ref, "SeriesDescription", "") or "")
        fs_blob = fluid.upper()
        fluid_sensitive = int(any(t in fs_blob for t in ("PDFS", "T2 FS", "T2FS", "STIR", "FLAIR", "FS", "FATSAT", "SPIR")))
        fat_suppression = int(any(t in fs_blob for t in ("FS", "FATSAT", "SPIR", "STIR", "FAT SAT", "FAT SUPP")))

        # Decode pixels
        arrays: List[np.ndarray] = []
        decode_errors = 0
        for ds in ds_list:
            try:
                arrays.append(dataset_to_float_array(ds))
            except RuntimeError:
                decode_errors += 1
        if decode_errors > len(ds_list) * 0.5 or not arrays:
            raise RuntimeError(
                f"{decode_errors}/{len(ds_list)} slices undecodable — install pylibjpeg-openjpeg/libjpeg wheels offline."
            )
        # Pad any skipped slices with zeros so depth stays consistent.
        while len(arrays) < len(ds_list):
            arrays.append(np.zeros_like(arrays[0]))

        vol_raw = np.stack(arrays, axis=0).astype(np.float32)  # (D,H,W)
        vol_u8 = percentile_normalize_stack(vol_raw, cfg.p_low, cfg.p_high)
        vol_resampled = resample_depth(vol_u8, cfg.num_slices)
        vol_final = resize_volume(vol_resampled, cfg.image_size)

        out_fp.parent.mkdir(parents=True, exist_ok=True)
        tmp_fp = out_fp.with_suffix(out_fp.suffix + ".tmp")
        if cfg.compress:
            np.savez_compressed(str(tmp_fp), arr=vol_final)
        else:
            np.save(str(tmp_fp), vol_final)
        os.replace(tmp_fp, out_fp)

        record.update({
            "status": "ok",
            "AnatomicalPlane": planes.title(),
            "FluidSensitive": fluid_sensitive,
            "FatSuppression": fat_suppression,
            "Rows": int(_safe_get(ref, "Rows", vol_final.shape[1])),
            "Columns": int(_safe_get(ref, "Columns", vol_final.shape[2])),
            "NumSlicesOriginal": int(len(ds_list)),
            "SortKeyUsed": sort_key,
            "TransferSyntaxUID": str(_safe_get(ref, "TransferSyntaxUID", "unknown")),
            "SeriesDescription": str(_safe_get(ref, "SeriesDescription", "")),
            "Seconds": round(time.time() - t0, 3),
        })
        return record
    except Exception as exc:  # noqa: BLE001
        record["status"] = "failed"
        record["error"] = f"{type(exc).__name__}: {exc}"
        record["traceback"] = traceback.format_exc(limit=3)
        logger.error("FAILED %s/%s: %s", study_uid, series_uid, exc)
        return record


# --------------------------------------------------------------------------------------
# Discovery & orchestration
# --------------------------------------------------------------------------------------
def discover_series(series_root: Path) -> List[Tuple[str, str, str]]:
    """Walk train_series/{StudyUID}/{SeriesUID}/ collecting (study, series, path) triples."""
    triples: List[Tuple[str, str, str]] = []
    root = Path(series_root)
    if not root.exists():
        raise FileNotFoundError(f"series root does not exist: {root}")
    for study_entry in os.scandir(str(root)):
        if not study_entry.is_dir():
            continue
        study_uid = study_entry.name
        for ser_entry in os.scandir(study_entry.path):
            if not ser_entry.is_dir():
                continue
            triples.append((study_uid, ser_entry.name, ser_entry.path))
    return triples


def run_preprocessing(
    series_root: str | Path,
    out_root: str | Path,
    manifest_path: str | Path,
    cfg: PreprocessConfig = PreprocessConfig(),
    limit: Optional[int] = None,
) -> "List[Dict[str, Any]]":
    """Main entrypoint: preprocess all series under `series_root` into `.npy` volumes + manifest."""
    series_root = Path(series_root)
    out_root = Path(out_root)
    out_root.mkdir(parents=True, exist_ok=True)

    triples = discover_series(series_root)
    if limit:
        triples = triples[:limit]
    logger.info("Discovered %d series under %s", len(triples), series_root)

    cfg_dict = asdict(cfg)
    cfg_dict["_out_root"] = str(out_root)
    jobs = [(s, u, p, cfg_dict) for (s, u, p) in triples]

    records: List[Dict[str, Any]] = []
    workers = cfg.max_workers if cfg.max_workers > 0 else max(1, (os.cpu_count() or 4))
    logger.info("Spawning process pool with %d workers (decoders:%s)", workers, _DECODER_HINT or " native-only")

    if workers == 1:
        for j in jobs:
            records.append(process_one_series(j))
    else:
        from multiprocessing import Pool
        with Pool(processes=workers, maxtasksperchild=32) as pool:
            for i, rec in enumerate(pool.imap_unordered(process_one_series, jobs, chunksize=4)):
                records.append(rec)
                if (i + 1) % 500 == 0 or (i + 1) == len(jobs):
                    ok = sum(r["status"] in ("ok", "skipped") for r in records)
                    logger.info("Progress: %d/%d series processed (ok=%d)", i + 1, len(jobs), ok)

    # Persist manifest (csv always; parquet best-effort).
    manifest_path = Path(manifest_path)
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    import csv
    fieldnames = ["StudyInstanceUID", "SeriesInstanceUID", "VolumePath", "status", "AnatomicalPlane",
                  "FluidSensitive", "FatSuppression", "Rows", "Columns", "NumSlicesOriginal",
                  "SortKeyUsed", "TransferSyntaxUID", "SeriesDescription", "Seconds", "error"]
    with open(manifest_path, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction="ignore")
        writer.writeheader()
        for r in records:
            writer.writerow(r)
    try:
        import pandas as pd
        df = pd.DataFrame(records)
        pq = manifest_path.with_suffix(".parquet")
        df.to_parquet(pq, index=False)
    except Exception:
        pass

    ok = sum(r["status"] in ("ok", "skipped") for r in records)
    failed = [r for r in records if r["status"] == "failed"]
    summary = {
        "total_series": len(records),
        "ok_or_skipped": ok,
        "failed": len(failed),
        "out_root": str(out_root),
        "manifest": str(manifest_path),
    }
    with open(out_root / "preprocess_summary.json", "w", encoding="utf-8") as fh:
        json.dump(summary, fh, indent=2)
    logger.info("Preprocessing complete: %s", summary)
    if failed:
        logger.warning("First failure example: %s", failed[0].get("error", "n/a"))
    return records


def main(argv: Optional[List[str]] = None) -> int:
    parser = argparse.ArgumentParser(description="RSNA Knee DICOM preprocessor (CPU, offline-safe)")
    parser.add_argument("--series-root", required=True, help="Root dir containing StudyUID/SeriesUID/*.dcm")
    parser.add_argument("--out-root", required=True, help="Where to write {study}_{series}.npy volumes")
    parser.add_argument("--manifest", required=True, help="Output manifest CSV path")
    parser.add_argument("--num-slices", type=int, default=16)
    parser.add_argument("--image-size", type=int, default=256)
    parser.add_argument("--p-low", type=float, default=0.5)
    parser.add_argument("--p-high", type=float, default=99.5)
    parser.add_argument("--compress", action="store_true", help="Save .npz compressed instead of .npy")
    parser.add_argument("--workers", type=int, default=0)
    parser.add_argument("--limit", type=int, default=None, help="Smoke-test cap on number of series")
    ns = parser.parse_args(argv)

    cfg = PreprocessConfig(
        num_slices=ns.num_slices,
        image_size=(ns.image_size, ns.image_size),
        p_low=ns.p_low,
        p_high=ns.p_high,
        compress=ns.compress,
        max_workers=ns.workers,
    )
    run_preprocessing(ns.series_root, ns.out_root, ns.manifest, cfg=cfg, limit=ns.limit)
    return 0


if __name__ == "__main__":
    pass  # driven from notebook cells


In [ ]:
# --- Stage 1: execute preprocessing ----------------------------------------------------
_cfg = PreprocessConfig(num_slices=16, image_size=(256, 256))
_limit = 240 if SMOKE_TEST else None
if os.path.isdir(TRAIN_SERIES):
    t0 = time.time()
    run_preprocessing(TRAIN_SERIES, VOL_ROOT, MANIFEST, cfg=_cfg, limit=_limit)
    print(f"[Stage 1] done in {(time.time()-t0)/60:.1f} min")
else:
    print("[Stage 1] train_series missing - skipped")

## Stage 2 · Module 2 source — `parse_reports.py`

In [ ]:
"""
MODULE 2 — Clinical Report Weak-Supervision Pseudo-Labeler.

Only a fraction of the RSNA training studies carry human-verified ground-truth labels,
but ALL training rows contain a free-text `Report`. This module implements a
deterministic, clinically-grounded rule-based NLP pipeline (regex + NegEx-style
negation scoping) to extract pseudo-labels for the 12 target abnormalities:

    ACL, MCL, Medial Meniscus, Lateral Meniscus, Medial OA, Lateral OA, PF OA,
    Effusion, Synovitis, Baker's, Contusion, Fracture

Outputs:
    train_pseudo_labels.csv  — expanded label file:
        * ground-truth rows preserved verbatim (label_source == "gt")
        * unlabeled rows filled with report-derived labels (label_source == "pseudo")
    pseudo_label_validation.txt / .json — F1 & accuracy vs the labeled subset.

Kaggle usage:
    !python parse_reports.py --train-csv /kaggle/input/.../train.csv --out-csv train_pseudo_labels.csv
"""

from __future__ import annotations

import argparse
import json
import re
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

TARGET_COLS: List[str] = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
    "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]


# --------------------------------------------------------------------------------------
# Condition lexicons: (target -> list of positive mention regexes)
# --------------------------------------------------------------------------------------
def _rx(*patterns: str) -> re.Pattern:
    """Compile alternation of lowercase patterns with word boundaries where sensible."""
    joined = "|".join(f"(?:{p})" for p in patterns)
    return re.compile(joined, flags=re.IGNORECASE | re.DOTALL)


CONDITION_PATTERNS: Dict[str, re.Pattern] = {
    # Ligaments
    "ACL": _rx(
        r"\bacl\b[^.;]{0,80}?(tear|disrupt|ruptur|sprain|avuls|mucoid|attenuat|insufficient|non[- ]visualiz)",
        r"(anterior cruciate ligament)[^.;]{0,80}?(tear|disrupt|ruptur|sprain|avuls|attenuat)",
        r"(acl graft)[^.;]{0,60}?(fail|loose|torn|ruptur)",
        r"(complete|partial|full[- ]thickness|high[- ]grade|low[- ]grade|grade [iI]{1,3})[^.;]{0,40}?\bacl\b",
        r"\bacl\b[^.;]{0,40}?(complete|partial|full[- ]thickness) tear",
    ),
    "MCL": _rx(
        r"\bmcl\b[^.;]{0,80}?(tear|sprain|ruptur|disrupt|edema along|thickening|increased signal)",
        r"(medial collateral ligament)[^.;]{0,80}?(tear|sprain|ruptur|disrupt|edema)",
        r"(edema along the (?:medial )?ligament)",
        r"\bmcl\b[^.;]{0,40}?grade [iI]{1,3}",
    ),
    # Menisci — require laterality adjacency to avoid cross-contamination.
    "Medial Meniscus": _rx(
        r"medial menisc[^.;]{0,80}?(tear|macera|complex tear|horizontal tear|radial tear|root tear|bucket|parameniscal|displac|degenerat|truncat)",
        r"(tear|macera|complex tear|horizontal tear|radial tear|root tear|bucket[- ]handle|parameniscal|degenerat\w* cyst|truncat\w*)[^.;]{0,40}?medial menisc",
        r"(medial)[^.;]{0,20}?(meniscal tear|meniscus tear)",
    ),
    "Lateral Meniscus": _rx(
        r"lateral menisc[^.;]{0,80}?(tear|macera|complex tear|horizontal tear|radial tear|root tear|bucket|parameniscal|displac|degenerat|truncat)",
        r"(tear|macera|complex tear|horizontal tear|radial tear|root tear|bucket[- ]handle|parameniscal|degenerat\w* cyst|truncat\w*)[^.;]{0,40}?lateral menisc",
        r"(lateral)[^.;]{0,20}?(meniscal tear|meniscus tear)",
    ),
    # Osteoarthritis compartments
    "Medial OA": _rx(
        r"(medial compartment\w*)[^.;]{0,60}?(narrow|joint space loss|osteoarthr|arthritis|cartilage loss|bone marrow lesion|subchondral)",
        r"(medial tibiofemoral)[^.;]{0,60}?(osteoarthr|arthritis|narrowing|cartilage loss)",
        r"(osteophyte[s]?)[^.;]{0,50}?medial",
        r"medial[^.;]{0,30}?(osteophyte[s]?|subchondral cyst|sclerosis)",
        r"(moderate|severe|mild)[^.;]{0,30}?(medial compartment|medial tibiofemoral) osteoarthr",
    ),
    "Lateral OA": _rx(
        r"(lateral compartment\w*)[^.;]{0,60}?(narrow|joint space loss|osteoarthr|arthritis|cartilage loss|bone marrow lesion|subchondral)",
        r"(lateral tibiofemoral)[^.;]{0,60}?(osteoarthr|arthritis|narrowing|cartilage loss)",
        r"(osteophyte[s]?)[^.;]{0,50}?lateral",
        r"lateral[^.;]{0,30}?(osteophyte[s]?|subchondral cyst|sclerosis)",
        r"(moderate|severe|mild)[^.;]{0,30}?(lateral compartment|lateral tibiofemoral) osteoarthr",
    ),
    "PF OA": _rx(
        r"(patellofemoral)[^.;]{0,60}?(osteoarthr|arthritis|arthrosis|cartilage loss|narrowing|spurr|osteophyte)",
        r"(trochlear groove)[^.;]{0,60}?(arthros|osteoarthr|cartilage loss|wear)",
        r"(patellar cartilage loss|patellar facet[^.;]{0,40}?(osteoarthr|cartilage loss))",
        r"(retropatellar)[^.;]{0,60}?(arthros|osteoarthr|cartilage loss|chondral)",
    ),
    # Fluid / synovium
    "Effusion": _rx(
        r"(joint effusion|suprapatellar effusion|knee joint effusion|large effusion|moderate effusion|small effusion|"
        r"joint fluid|fluid distension of the (?:joint|suprapatellar bursa)|effusion)",
    ),
    "Synovitis": _rx(
        r"(synovitis|synovial thickening|synovial proliferation|synovial enhancement|hypertrophic synovium|villonodular)",
    ),
    "Baker's": _rx(
        r"(baker'?s? cyst|popliteal cyst|semimembranosus[- ]gastrocnemius bursitis|"
        r"gastrocnemius[- ]semimembranosus bursitis|cyst in the popliteal fossa)",
    ),
    # Bone
    "Contusion": _rx(
        r"(bone contusion|bone bruise|bone marrow edema|bone marrow oedema|subchondral edema|subchondral fracture[^.;]{0,40}?(edema)?|"
        r"marrow edema pattern|post[- ]traumatic edema[^.;]{0,40}(femur|tibia))",
        r"(femoral|tibial|patellar)[^.;]{0,30}?(contusion|bone bruise|marrow edema)",
    ),
    "Fracture": _rx(
        r"(fractur\w*|cortical breach|cortical step[- ]?off|avulsion(?: fracture)?|displaced fragment|break in the cortex)",
        r"(tibial plateau|femoral condyle|fibular head|patella)[^.;]{0,40}?(break|depress\w*|comminut\w*|non[- ]?displaced fracture)",
    ),
}

NEGATION_CUES: List[str] = [
    "no evidence of", "no evidence for", "without", "free of", "no sign of", "no signs of",
    "intact", "unremarkable", "normal", "negative for", "not present", "no acute",
    "no demonstrative", "no", "none", "denies", "clear", "preserved", "grossly intact",
    "no significant", "without evidence", "rules out", "excluded",
]

# Positive-triggers that *cancel* a preceding negation within scope (NegEx "trigger").
CANCELLED_CUES: List[str] = [
    "but", "however", "although", "though", "still", "nevertheless", "except", "cannot exclude",
    "may represent", "possible", "suspected", "question of",
]

# Precomputed compiled cue sets
_NEG_RX = re.compile(r"\b(" + "|".join(re.escape(c) for c in NEGATION_CUES) + r")\b", re.I)
_CANCEL_RX = re.compile(r"\b(" + "|".join(re.escape(c) for c in CANCELLED_CUES) + r")\b", re.I)

# Scope windows (in characters) searched before the matched condition mention for neg cues.
NEG_SCOPE_BEFORE = 60
NEG_SCOPE_AFTER = 25


@dataclass
class PseudoLabelResult:
    labels: Dict[str, int]          # target -> 0/1
    confidence: Dict[str, float]    # target -> confidence in [0,1]
    mentions: Dict[str, List[str]]  # target -> matched snippets (audit trail)


def _is_negated(sentence: str, start: int, end: int) -> bool:
    """NegEx-lite: look for a negation cue immediately before the mention, and make sure
    no cancellation cue appears between the cue and the mention."""
    pre = sentence[max(0, start - NEG_SCOPE_BEFORE):start]
    post = sentence[end:end + NEG_SCOPE_AFTER]
    m = None
    for m in _NEG_RX.finditer(pre):
        pass  # take the LAST (closest) neg cue before the mention
    if m is None:
        # some cues may follow ("... is intact")
        pm = _NEG_RX.search(post)
        if pm is not None and not _CANCEL_RX.search(post[:pm.start()]):
            return True
        return False
    between = pre[m.end():]
    if _CANCEL_RX.search(between):
        return False
    # A sentence boundary (period) between cue and mention breaks negation scope.
    if "." in between and len(between.strip()) > 0 and between.strip().startswith("."):
        return False
    return True


def extract_pseudo_labels_from_report(report_text: str) -> PseudoLabelResult:
    """Rule-based multi-label extraction from one radiology report.

    Args:
        report_text: Free-text MRI knee radiology report. May be empty/NaN.
    Returns:
        PseudoLabelResult with binary labels, per-target confidence, audit snippets.
    """
    labels: Dict[str, int] = {t: 0 for t in TARGET_COLS}
    conf: Dict[str, float] = {t: 0.0 for t in TARGET_COLS}
    mentions: Dict[str, List[str]] = {t: [] for t in TARGET_COLS}

    if report_text is None or (isinstance(report_text, float) and np.isnan(report_text)):
        return PseudoLabelResult(labels, conf, mentions)
    text = str(report_text)
    if not text.strip():
        return PseudoLabelResult(labels, conf, mentions)

    # Split into sentences for local negation scoping while keeping offsets.
    sentences: List[Tuple[int, str]] = []
    pos = 0
    for chunk in re.split(r"(?<=[.;])\s+", text):
        idx = text.find(chunk, pos)
        if idx < 0:
            idx = pos
        sentences.append((idx, chunk))
        pos = idx + len(chunk)

    for target, rx in CONDITION_PATTERNS.items():
        n_pos = 0
        n_neg = 0
        for sent_start, sent in sentences:
            for m in rx.finditer(sent):
                snippet = m.group(0)[:120].strip()
                if _is_negated(sent, m.start(), m.end()):
                    n_neg += 1
                else:
                    n_pos += 1
                    if len(mentions[target]) < 3:
                        mentions[target].append(snippet)
        if n_pos > 0:
            labels[target] = 1
            # Confidence grows with number of independent positive mentions; negations reduce it.
            base = min(1.0, 0.7 + 0.1 * (n_pos - 1))
            conf[target] = round(base * (1.0 - 0.25 * min(n_neg, 2)), 3)
        else:
            labels[target] = 0
            # High-confidence negatives only when the anatomy was discussed but negated.
            discussed = any(rx.search(s) for _, s in sentences)
            conf[target] = round(0.85 if (discussed and n_neg > 0) else 0.6, 3)
    return PseudoLabelResult(labels, conf, mentions)


# --------------------------------------------------------------------------------------
# Validation against ground truth
# --------------------------------------------------------------------------------------
def validate_pseudo_labels(gt_df: pd.DataFrame, reports: pd.Series, threshold: float = 0.6) -> Dict[str, object]:
    """Compare extracted pseudo-labels against available ground-truth rows.

    Args:
        gt_df: DataFrame containing StudyInstanceUID + the 12 TARGET_COLS (0/1).
        reports: Series indexed identically to gt_df with the report text.
        threshold: minimum confidence required to accept a pseudo-label as "positive".
    Returns:
        dict with per-target and macro precision/recall/F1/accuracy.
    """
    from sklearn.metrics import f1_score, accuracy_score, roc_auc_score

    y_true_all: List[List[int]] = []
    y_pred_all: List[List[int]] = []
    per_target_stats: Dict[str, Dict[str, float]] = {}

    aligned = gt_df.reset_index(drop=True)
    rep = reports.reindex(aligned.index).reset_index(drop=True) if hasattr(reports, "reindex") else pd.Series(list(reports))

    for i in range(len(aligned)):
        res = extract_pseudo_labels_from_report(rep.iloc[i])
        yt = [int(aligned.iloc[i][t]) for t in TARGET_COLS]
        yp = [1 if (res.labels[t] == 1 and res.confidence[t] >= threshold) else 0 for t in TARGET_COLS]
        y_true_all.append(yt)
        y_pred_all.append(yp)

    y_true_np = np.array(y_true_all, dtype=np.int8)
    y_pred_np = np.array(y_pred_all, dtype=np.int8)

    for j, t in enumerate(TARGET_COLS):
        try:
            f1_t = float(f1_score(y_true_np[:, j], y_pred_np[:, j], zero_division=0))
        except Exception:
            f1_t = float("nan")
        acc_t = float(accuracy_score(y_true_np[:, j], y_pred_np[:, j]))
        per_target_stats[t] = {"f1": f1_t, "accuracy": acc_t,
                               "prevalence_gt": float(y_true_np[:, j].mean()),
                               "prevalence_pred": float(y_pred_np[:, j].mean())}

    macro_f1 = float(f1_score(y_true_np, y_pred_np, average="macro", zero_division=0))
    micro_f1 = float(f1_score(y_true_np, y_pred_np, average="micro", zero_division=0))
    exact_acc = float(np.mean(np.all(y_true_np == y_pred_np, axis=1)))
    sample_hamming = float(np.mean(np.sum(y_true_np != y_pred_np, axis=1)) / len(TARGET_COLS))

    # AUC proxy using confidence-weighted positives where computable.
    auc_scores: List[float] = []
    for j, t in enumerate(TARGET_COLS):
        scores = np.array([extract_pseudo_labels_from_report(rep.iloc[i]).confidence[t] *
                           extract_pseudo_labels_from_report(rep.iloc[i]).labels[t]
                           for i in range(min(len(rep), 300))])
        try:
            if len(np.unique(y_true_np[:len(scores), j])) > 1:
                auc_scores.append(float(roc_auc_score(y_true_np[:len(scores), j], scores)))
        except Exception:
            pass
    mean_auc_proxy = float(np.mean(auc_scores)) if auc_scores else float("nan")

    summary = {
        "n_validated": int(len(y_true_np)),
        "per_target": per_target_stats,
        "macro_f1": macro_f1,
        "micro_f1": micro_f1,
        "exact_match_accuracy": exact_acc,
        "hamming_loss_per_label": sample_hamming,
        "mean_auc_confidence_proxy": mean_auc_proxy,
        "threshold_used": threshold,
    }
    return summary


# --------------------------------------------------------------------------------------
# Expansion pipeline: GT + pseudo-labels
# --------------------------------------------------------------------------------------
def build_expanded_labels(
    train_csv_path: str | Path,
    out_csv_path: str | Path,
    validation_out_prefix: Optional[str | Path] = None,
    confidence_threshold: float = 0.6,
    max_pseudo_rows: Optional[int] = None,
) -> pd.DataFrame:
    """Read train.csv, keep ground-truth rows untouched, fill unlabeled rows with report-derived labels.

    The official RSNA release marks unlabeled rows by NaN in the target columns while still
    providing the Report text. Rows whose entire label vector is NaN become pseudo-labeled.
    """
    df = pd.read_csv(train_csv_path)
    if "StudyInstanceUID" not in df.columns:
        raise ValueError("train.csv must contain StudyInstanceUID column")
    has_report = "Report" in df.columns
    if not has_report:
        raise ValueError("train.csv must contain the free-text 'Report' column for weak supervision")

    labeled_mask = df[TARGET_COLS].notna().any(axis=1)
    gt_df = df.loc[labeled_mask, ["StudyInstanceUID"] + TARGET_COLS].copy()
    unl_df = df.loc[~labeled_mask, ["StudyInstanceUID", "Report"]].copy()

    logger_lines: List[str] = []
    logger_lines.append(f"Ground-truth labeled studies : {len(gt_df)}")
    logger_lines.append(f"Unlabeled studies w/ reports : {len(unl_df)}")

    # --- Validate parser on the labeled subset first -------------------------------
    val_summary: Dict[str, object] = {}
    if len(gt_df) > 0:
        reports_for_gt = df.loc[labeled_mask, "Report"].reset_index(drop=True)
        try:
            val_summary = validate_pseudo_labels(gt_df, reports_for_gt, threshold=confidence_threshold)
            logger_lines.append(
                f"Parser validation on GT subset: macro-F1={val_summary.get('macro_f1', float('nan')):.3f}, "
                f"exact-match-acc={val_summary.get('exact_match_accuracy', float('nan')):.3f}"
            )
        except ImportError:
            logger_lines.append("sklearn unavailable in this environment; skipped validation metrics.")

    # --- Generate pseudo labels -----------------------------------------------------
    if max_pseudo_rows is not None:
        unl_df = unl_df.head(max_pseudo_rows)

    rows: List[Dict[str, object]] = []
    for uid, rpt in zip(unl_df["StudyInstanceUID"].values, unl_df["Report"].values):
        res = extract_pseudo_labels_from_report("" if pd.isna(rpt) else str(rpt))
        row: Dict[str, object] = {"StudyInstanceUID": uid, "label_source": "pseudo"}
        for t in TARGET_COLS:
            lab = res.labels[t]
            cnf = res.confidence[t]
            # Only trust positives above threshold; keep confident negatives as 0.
            final_lab = 1 if (lab == 1 and cnf >= confidence_threshold) else 0
            row[t] = final_lab
        row["pseudo_mean_conf"] = float(np.mean(list(res.confidence.values())))
        rows.append(row)
    pseudo_df = pd.DataFrame(rows, columns=["StudyInstanceUID", "label_source"] + TARGET_COLS + ["pseudo_mean_conf"])

    gt_out = gt_df.copy()
    gt_out["label_source"] = "gt"
    combined = pd.concat([gt_out, pseudo_df], ignore_index=True)
    combined = combined.drop_duplicates(subset="StudyInstanceUID", keep="first")

    out_csv_path = Path(out_csv_path)
    out_csv_path.parent.mkdir(parents=True, exist_ok=True)
    combined.to_csv(out_csv_path, index=False)
    logger_lines.append(f"Wrote expanded labels: {out_csv_path} ({len(combined)} rows)")

    if validation_out_prefix is not None:
        prefix = Path(validation_out_prefix)
        prefix.parent.mkdir(parents=True, exist_ok=True)
        with open(prefix.with_suffix(".txt"), "w", encoding="utf-8") as fh:
            fh.write("\n".join(logger_lines) + "\n\n")
            fh.write(json.dumps(val_summary, indent=2, default=float))
        with open(prefix.with_suffix(".json"), "w", encoding="utf-8") as fh:
            json.dump({"log": logger_lines, "validation": val_summary}, fh, indent=2, default=float)

    print("\n".join(logger_lines))
    return combined


def main(argv: Optional[List[str]] = None) -> int:
    ap = argparse.ArgumentParser(description="RSNA Knee report weak-supervision pseudo-labeler")
    ap.add_argument("--train-csv", required=True, help="Path to train.csv (with Report column)")
    ap.add_argument("--out-csv", required=True, help="Output expanded labels csv")
    ap.add_argument("--validation-prefix", default=None, help="Write validation report txt/json here")
    ap.add_argument("--confidence-threshold", type=float, default=0.6)
    ap.add_argument("--max-pseudo-rows", type=int, default=None)
    ns = ap.parse_args(argv)

    build_expanded_labels(
        train_csv_path=ns.train_csv,
        out_csv_path=ns.out_csv,
        validation_out_prefix=ns.validation_prefix,
        confidence_threshold=ns.confidence_threshold,
        max_pseudo_rows=ns.max_pseudo_rows,
    )
    return 0


if __name__ == "__main__":
    pass  # driven from notebook cells


In [ ]:
# --- Stage 2: pseudo-label expansion -----------------------------------------------------
if os.path.exists(TRAIN_CSV):
    build_expanded_labels(TRAIN_CSV, PSEUDO_CSV,
                          validation_out_prefix=os.path.join(WORK, "pseudo_val"),
                          confidence_threshold=0.6,
                          max_pseudo_rows=1500 if SMOKE_TEST else None)
else:
    print("[Stage 2] train.csv missing - skipped")

## Stage 3 · Module 3 source — `train.py`

In [ ]:
"""
MODULE 3 — Multi-View Attention-Pooled 2.5D Model & Training Loop (RSNA Knee).

Contents
--------
* Config dataclass (SEED/IMAGE_SIZE/NUM_SLICES/BATCH_SIZE/BACKBONE/TARGET_COLS/DEVICE/MIXED_PRECISION...)
* RSNAKneeDataset      : study-level dataset over preprocessed .npy volumes (Module 1 output),
                         picks up to 3 dominant series per anatomical plane (Sagittal/Coronal/Axial),
                         zero-pads missing planes; consistent-across-slices Albumentations augmentation.
* RSNAMultiPlaneModel  : timm 2D backbone applied slice-wise (2.5D), BiGRU / Multi-Head Self-Attention
                         aggregator + attention pooling across slices, plane concatenation,
                         metadata embeddings (FluidSensitive/FatSuppression/plane) fused before head,
                         12-logit multi-label classifier with sigmoid at inference.
* AsymmetricLoss       : ASL (gamma_neg=4, gamma_pos=0, clip=0.05) optimized implementation.
* MacroAUC             : competition metric = mean of per-class ROC-AUC, crash-safe on single-class batches.
* Warmup+Cosine LR schedule, AdamW, torch.amp autocast, EMA weights, GroupKFold(5) by StudyInstanceUID,
  OOF macro-AUC every epoch, best checkpoint per fold (`best_auc_fold_{k}.pt`).

Run (Kaggle):
    !python train.py --backbone convnext_nano.in12k_ft_in1k --fold 0 --epochs 6 \
        --volumes-root /kaggle/input/rsna-knee-preprocessed-256/train_volumes \
        --manifest /kaggle/working/train_manifest.csv \
        --labels /kaggle/working/train_pseudo_labels.csv \
        --ckpt-dir /kaggle/working/checkpoints
"""

from __future__ import annotations

import argparse
import json
import math
import os
import random
import sys
import time
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

TARGET_COLS: List[str] = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
    "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]
PLANES: List[str] = ["Sagittal", "Coronal", "Axial"]


# --------------------------------------------------------------------------------------
# SECTION 1: SYSTEM CONFIG & HYPERPARAMETERS
# --------------------------------------------------------------------------------------
@dataclass
class Config:
    SEED: int = 42
    IMAGE_SIZE: Tuple[int, int] = (256, 256)
    NUM_SLICES: int = 16
    BATCH_SIZE: int = 8
    NUM_WORKERS: int = 4
    BACKBONE: str = "convnext_nano.in12k_ft_in1k"   # or "tf_efficientnet_b0_ns" for memory efficiency
    IN_CHANNELS: int = 3                            # 3-channel duplicate for pretrained weights
    NUM_PLANES: int = 3
    FEATURE_DIM: int = 256                          # projection dim after backbone
    AGGREGATOR: str = "bigru"                       # {"bigru", "mhsa"}
    ATTENTION_HEADS: int = 4
    DROPOUT: float = 0.25
    LR: float = 3e-4
    WEIGHT_DECAY: float = 1e-2
    EPOCHS: int = 6
    WARMUP_EPOCHS: int = 1
    N_FOLDS: int = 5
    MIXED_PRECISION: bool = True
    USE_EMA: bool = True
    EMA_DECAY: float = 0.995
    PSEUDO_WEIGHT: float = 0.5                      # loss weight for pseudo-labeled rows
    TARGET_COLS: List[str] = field(default_factory=lambda: list(TARGET_COLS))
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"


def set_determinism(seed: int) -> None:
    """Strict determinism across Python / NumPy / PyTorch / CuDNN."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# --------------------------------------------------------------------------------------
# SECTION 4: DATASET (study-level, multi-plane, consistent slice augmentation)
# --------------------------------------------------------------------------------------
def load_volume(path: str) -> np.ndarray:
    """Load a preprocessed (D,H,W) uint8 volume from .npy or compressed .npz."""
    if path.endswith(".npz"):
        with np.load(path) as z:
            arr = z["arr"] if "arr" in z.files else z[z.files[0]]
    else:
        arr = np.load(path)
    return np.asarray(arr)


class ConsistentSliceAug:
    """Applies identical geometric augmentation params across all slices of one series.

    Uses albumentations when available; otherwise falls back to pure-numpy ops so the
    pipeline never hard-fails on Kaggle images lacking the package.
    """

    def __init__(self, p_flip: float = 0.5, p_affine: float = 0.7, p_brightness: float = 0.5,
                 p_dropout: float = 0.3, image_size: Tuple[int, int] = (256, 256), seed: int = 42):
        self.p_flip, self.p_affine, self.p_brightness, self.p_dropout = p_flip, p_affine, p_brightness, p_dropout
        self.image_size = image_size
        self.rng = np.random.default_rng(seed)
        try:
            import albumentations as A
            self.transform = A.Compose([
                A.HorizontalFlip(p=p_flip),
                A.ShiftScaleRotate(shift_limit=0.06, scale_limit=0.10, rotate_limit=10, p=p_affine,
                                   border_mode=0, value=0),
                A.RandomBrightnessContrast(brightness_limit=0.15, contrast_limit=0.15, p=p_brightness),
                A.CoarseDropout(max_holes=6, max_height=24, max_width=24, min_holes=1,
                                fill_value=0, p=p_dropout),
            ])
        except Exception:
            self.transform = None

    def __call__(self, vol: np.ndarray) -> np.ndarray:
        """vol: (D,H,W) uint8 -> augmented (D,H,W) uint8 with SHARED geometry."""
        d = vol.shape[0]
        if self.transform is not None:
            # Draw ONE geometric realization and apply identically to each slice by
            # seeding the underlying RNG through repeated calls on a reference patch.
            ref = self._consistent_apply(vol)
            return ref
        # ---- numpy fallback -----------------------------------------------------
        out = vol.copy()
        if self.rng.random() < self.p_flip:
            out = out[:, :, ::-1]
        if self.rng.random() < self.p_dropout:
            h, w = out.shape[1], out.shape[2]
            n_holes = int(self.rng.integers(1, 7))
            for _ in range(n_holes):
                hh = int(self.rng.integers(6, 25)); ww = int(self.rng.integers(6, 25))
                y0 = int(self.rng.integers(0, max(1, h - hh))); x0 = int(self.rng.integers(0, max(1, w - ww)))
                out[:, y0:y0 + hh, x0:x0 + ww] = 0
        if self.rng.random() < self.p_brightness:
            b = float(self.rng.uniform(-0.12, 0.12))
            out = np.clip(out.astype(np.float32) * (1.0 + b), 0, 255).astype(np.uint8)
        return out

    def _consistent_apply(self, vol: np.ndarray) -> np.ndarray:
        """Albumentations applies per-image randomness; we force consistency by using
        a keypoint-free deterministic seed derived per-series call."""
        import albumentations as A
        seed = int(self.rng.integers(0, 2**31 - 1))
        A.set_random_seed(seed)
        frames = []
        for s in vol:
            res = self.transform(image=s)
            frames.append(res["image"])
        return np.stack(frames, axis=0)


class RSNAKneeDataset(Dataset):
    """Study-level dataset. Each item = up to 3 plane volumes + metadata + label vector.

    Returns tensors shaped:
        volumes : (NUM_PLANES, NUM_SLICES, H, W) float32 in [0,1]
        plane_mask : (NUM_PLANES,) float32 (1 if that plane exists)
        meta : (4,) float32 [any_fluid_sensitive, any_fat_suppressed, n_series_norm, has_axial]
        target : (12,) float32
        sample_weight : scalar tensor (1.0 GT / cfg.PSEUDO_WEIGHT pseudo)
    """

    def __init__(
        self,
        studies_df: pd.DataFrame,
        manifest_df: pd.DataFrame,
        volumes_root: str | Path,
        cfg: Config,
        train: bool = True,
    ):
        self.cfg = cfg
        self.volumes_root = Path(volumes_root)
        self.train = train
        self.studies = studies_df.reset_index(drop=True)

        # Index manifest by study
        m = manifest_df.copy()
        m = m[m["status"].isin(["ok", "skipped"])].copy()
        self._by_study: Dict[str, List[Dict[str, object]]] = {}
        for rec in m.to_dict(orient="records"):
            self._by_study.setdefault(rec["StudyInstanceUID"], []).append(rec)

        # If skip_existing left planes unknown, fall back to SeriesDescription parsing.
        self.aug = ConsistentSliceAug(image_size=cfg.IMAGE_SIZE, seed=cfg.SEED) if train else None

        # Pre-compute per-study chosen series (deterministic selection).
        self._chosen: List[List[Optional[Dict[str, object]]]] = []
        for uid in self.studies["StudyInstanceUID"].values:
            self._chosen.append(self._select_series(str(uid)))

    # ------------------------------------------------------------------
    def _plane_of(self, rec: Dict[str, object]) -> str:
        plane = str(rec.get("AnatomicalPlane", "") or "").lower()
        desc = str(rec.get("SeriesDescription", "") or "").lower()
        for canonical, keys in (("sagittal", ("sag",)), ("coronal", ("cor",)), ("axial", ("ax", "tra"))):
            if any(k in plane for k in keys) or any(f"{k}" in desc for k in keys):
                return canonical.capitalize() if False else canonical
        return "unknown"

    def _select_series(self, uid: str) -> List[Optional[Dict[str, object]]]:
        """Pick up to one dominant series per plane (Sagittal, Coronal, Axial).

        Dominance heuristic: most slices, tie-break preferring fluid-sensitive sequences.
        """
        cands = self._by_study.get(uid, [])
        best: Dict[str, Dict[str, object]] = {}
        for rec in cands:
            pl = self._plane_of(rec)
            if pl == "unknown":
                continue
            cur = best.get(pl)
            score = (int(rec.get("NumSlicesOriginal", 0) or 0), int(rec.get("FluidSensitive", 0) or 0))
            if cur is None:
                best[pl] = rec
            else:
                cur_score = (int(cur.get("NumSlicesOriginal", 0) or 0), int(cur.get("FluidSensitive", 0) or 0))
                if score > cur_score:
                    best[pl] = rec
        ordered: List[Optional[Dict[str, object]]] = []
        for pl in ("sagittal", "coronal", "axial"):
            ordered.append(best.get(pl))
        # Degenerate case: no recognized planes at all -> take first 2 readable series as sag/cor slots.
        if all(o is None for o in ordered) and cands:
            ordered = [cands[0], cands[1] if len(cands) > 1 else None, None]
        return ordered

    # ------------------------------------------------------------------
    def __len__(self) -> int:
        return len(self.studies)

    def _load_plane_volume(self, rec: Optional[Dict[str, object]]) -> np.ndarray:
        if rec is None:
            return np.zeros((self.cfg.NUM_SLICES, *self.cfg.IMAGE_SIZE), dtype=np.uint8)
        path = self.volumes_root / Path(str(rec["VolumePath"])).name
        try:
            vol = load_volume(str(path))
        except Exception:
            # direct stored absolute path attempt
            try:
                vol = load_volume(str(rec["VolumePath"]))
            except Exception:
                return np.zeros((self.cfg.NUM_SLICES, *self.cfg.IMAGE_SIZE), dtype=np.uint8)
        # Defensive shape repair: enforce exactly K slices and HxW.
        if vol.ndim != 3:
            vol = vol.reshape(-1, vol.shape[-2], vol.shape[-1]) if vol.ndim > 3 else vol[None].repeat(self.cfg.NUM_SLICES, 0)
        if vol.shape[0] != self.cfg.NUM_SLICES:
            idx = np.linspace(0, vol.shape[0] - 1, self.cfg.NUM_SLICES).round().astype(int)
            vol = vol[idx]
        if vol.shape[1:] != tuple(self.cfg.IMAGE_SIZE):
            try:
                import cv2
                vol = np.stack([cv2.resize(v, tuple(reversed(self.cfg.IMAGE_SIZE)), interpolation=cv2.INTER_AREA) for v in vol])
            except Exception:
                from PIL import Image
                vol = np.stack([np.asarray(Image.fromarray(v).resize(tuple(reversed(self.cfg.IMAGE_SIZE)))) for v in vol])
        return vol.astype(np.uint8, copy=False)

    def __getitem__(self, index: int) -> Dict[str, torch.Tensor]:
        row = self.studies.iloc[index]
        uid = str(row["StudyInstanceUID"])
        chosen = self._chosen[index]

        vols, masks, fluids, fats = [], [], [], []
        for rec in chosen:
            v = self._load_plane_volume(rec)
            if self.aug is not None and rec is not None:
                v = self.aug(v)
            vols.append(v)
            masks.append(1.0 if rec is not None else 0.0)
            fluids.append(float(rec.get("FluidSensitive", 0) or 0) if rec else 0.0)
            fats.append(float(rec.get("FatSuppression", 0) or 0) if rec else 0.0)

        vol_stack = np.stack(vols, axis=0).astype(np.float32) / 255.0     # (P,D,H,W)
        vol_stack = np.repeat(vol_stack[:, :, None], self.cfg.IN_CHANNELS, axis=2)  # (P,D,C,H,W)

        labels = np.array([float(row[t]) if pd.notna(row[t]) else 0.0 for t in TARGET_COLS], dtype=np.float32)
        src = str(row.get("label_source", "gt")).lower()
        weight = 1.0 if src == "gt" else self.cfg.PSEUDO_WEIGHT

        meta = np.array([
            float(np.max(fluids)), float(np.max(fats)),
            float(sum(masks)) / 3.0, float(masks[2]),
        ], dtype=np.float32)

        return {
            "volumes": torch.from_numpy(vol_stack),                     # (P,D,C,H,W)
            "plane_mask": torch.tensor(masks, dtype=torch.float32),     # (P,)
            "fluid_flags": torch.tensor(fluids, dtype=torch.float32),   # (P,)
            "fat_flags": torch.tensor(fats, dtype=torch.float32),       # (P,)
            "meta": torch.from_numpy(meta),
            "target": torch.from_numpy(labels),
            "weight": torch.tensor(weight, dtype=torch.float32),
            "uid": uid,
        }


def collate_knee(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
    uids = [b.pop("uid") for b in batch]
    out = {k: torch.stack([b[k] for b in batch], dim=0) for k in
           ["volumes", "plane_mask", "fluid_flags", "fat_flags", "meta", "target", "weight"]}
    out["uids"] = uids  # type: ignore[assignment]
    return out


# --------------------------------------------------------------------------------------
# SECTION 5: MODEL ARCHITECTURE
# --------------------------------------------------------------------------------------
def build_backbone(name: str, in_chans: int) -> Tuple[nn.Module, int]:
    """Create a timm backbone returning (features, embed_dim). Graceful offline handling.

    If `timm` is not importable at all (minimal container), falls back to the local CNN
    so the pipeline never hard-crashes; pretrained weights are only attempted when timm
    is present, and failures degrade to random-init architecture instead of aborting.
    """
    errors: List[str] = []
    try:
        import timm
        # Try exact name first, then without tag suffix (offline caches often keyed differently).
        for cand in (name, name.split(".")[0]):
            for pre in (True, False):
                try:
                    m = timm.create_model(cand, pretrained=pre, in_chans=in_chans, num_classes=0)
                    dim = getattr(m, "num_features", None)
                    if dim is None:
                        with torch.no_grad():
                            probe = m(torch.zeros(1, in_chans, 64, 64))
                            dim = probe.shape[-1] if probe.ndim == 2 else probe.shape[1]
                    if not pre:
                        print(f"[WARN] Using random-init {cand} (weights download unavailable/offline).")
                    return m, int(dim)
                except Exception as exc:  # noqa: BLE001
                    errors.append(f"{cand}(pretrained={pre}): {exc}")
    except ImportError as exc:
        errors.append(f"timm import failed: {exc}")
    # Last resort: tiny local CNN so submission never dies because timm is missing.
    print("[WARN] Could not load timm backbone; falling back to lightweight local CNN.", errors[:2])

    class FallbackCNN(nn.Module):
        def __init__(self, c: int, dim: int = 512):
            super().__init__()
            def block(ci, co):
                return nn.Sequential(nn.Conv2d(ci, co, 3, stride=2, padding=1, bias=False),
                                     nn.BatchNorm2d(co), nn.SiLU(inplace=True))
            self.net = nn.Sequential(block(c, 32), block(32, 64), block(64, 128), block(128, 256),
                                     nn.AdaptiveAvgPool2d(1), nn.Flatten())
            self.num_features = dim

        def forward(self, x):
            return self.net(x)

    dim = 256
    return FallbackCNN(in_chans, dim), dim


class SliceAggregator(nn.Module):
    """Bidirectional GRU or Multi-Head Self-Attention + attention pooling over slices."""

    def __init__(self, dim: int, mode: str = "bigru", heads: int = 4, dropout: float = 0.25):
        super().__init__()
        self.mode = mode
        if mode == "bigru":
            self.gru = nn.GRU(dim, dim // 2, num_layers=1, batch_first=True,
                              bidirectional=True, dropout=dropout)
        else:  # mhsa
            self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, dropout=dropout, batch_first=True)
            self.norm1 = nn.LayerNorm(dim)
        # Learned query attention pooling -> study-level vector
        self.pool_query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.pool_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=max(1, heads // 2), batch_first=True, dropout=dropout)
        self.norm = nn.LayerNorm(dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """x: (B, D, dim) -> (B, dim)"""
        if self.mode == "bigru":
            h, _ = self.gru(x)
            h = self.norm1(h) if hasattr(self, "norm1") else h
        else:
            attn_mask = None
            if mask is not None:
                attn_mask = ~mask.bool()          # True => ignore
                if attn_mask.all(dim=1).any():    # fully-masked rows would NaN softmax
                    attn_mask = None
            att, _ = self.attn(x, x, x, attn_mask=attn_mask)
            h = self.norm1(x + att)
        q = self.pool_query.expand(x.shape[0], -1, -1)
        key_mask = None
        if mask is not None:
            key_mask = ~mask.bool()
            if key_mask.all(dim=1).any():
                key_mask = None
        pooled, _ = self.pool_attn(q, h, h, key_padding_mask=key_mask, need_weights=False)
        return self.drop(self.norm(pooled.squeeze(1)))


class RSNAMultiPlaneModel(nn.Module):
    """Shared 2.5D backbone across planes + per-plane aggregators + metadata fusion."""

    def __init__(self, cfg: Config):
        super().__init__()
        self.cfg = cfg
        backbone, feat_dim = build_backbone(cfg.BACKBONE, cfg.IN_CHANNELS)
        self.backbone = backbone
        self.proj = nn.Sequential(nn.Linear(feat_dim, cfg.FEATURE_DIM), nn.ReLU(inplace=True))

        self.aggregators = nn.ModuleList([
            SliceAggregator(cfg.FEATURE_DIM, mode=cfg.AGGREGATOR, heads=cfg.ATTENTION_HEADS, dropout=cfg.DROPOUT)
            for _ in range(cfg.NUM_PLANES)
        ])

        # Metadata embeddings: FluidSensitive, FatSuppression (per plane) + plane indicator.
        self.meta_embed = nn.Embedding(4, 16)   # 0=none(pad),1=sag,2=cor,3=ax combined w/ flags handled separately
        self.flags_proj = nn.Linear(3 * 2, 32)  # per-plane [fluid, fat] flags flattened
        self.plane_embed = nn.Embedding(cfg.NUM_PLANES, 8)

        fused_dim = cfg.FEATURE_DIM * cfg.NUM_PLANES + 32 + cfg.NUM_PLANES * 8
        self.head = nn.Sequential(
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(fused_dim, 512),
            nn.GELU(),
            nn.Dropout(cfg.DROPOUT),
            nn.Linear(512, len(cfg.TARGET_COLS)),
        )

    def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
        """batch['volumes']: (B,P,D,C,H,W) -> logits (B,12)"""
        vols = batch["volumes"]
        b, p, d, c, h, w = vols.shape
        x = vols.reshape(b * p * d, c, h, w)
        with torch.autocast(device_type=self.cfg.DEVICE, dtype=torch.float16,
                            enabled=self.cfg.MIXED_PRECISION and self.cfg.DEVICE == "cuda"):
            feats = self.backbone(x)                       # (B*P*D, feat_dim)
            feats = self.proj(feats.float())               # (B*P*D, FEATURE_DIM) in fp32
        feats = feats.reshape(b * p, d, -1)

        plane_masks = batch["plane_mask"].reshape(b * p)   # (B*P,)
        outs = []
        for pi in range(p):
            sel = slice(pi, b * p, p)                      # indices for plane pi across batch
            fp = feats[sel]                                # (B, D, FEATURE_DIM)
            mp = plane_masks[sel]                          # (B,)
            agg = self.aggregators[pi](fp, mask=None)      # shapes fixed; pad volumes are zeros
            # Zero-out contributions of missing planes.
            agg = agg * mp.unsqueeze(-1)
            outs.append(agg)

        plane_feats = torch.cat(outs, dim=-1)              # (B, P*FEATURE_DIM)
        flags = torch.cat([batch["fluid_flags"], batch["fat_flags"]], dim=-1)  # (B, 2P)
        flag_emb = self.flags_proj(flags)
        plane_ids = torch.arange(p, device=vols.device).unsqueeze(0).expand(b, -1)
        p_emb = self.plane_embed(plane_ids).reshape(b, -1)
        fused = torch.cat([plane_feats, flag_emb, p_emb], dim=-1)
        logits = self.head(fused)
        return logits


# --------------------------------------------------------------------------------------
# SECTION 6: LOSS & METRICS
# --------------------------------------------------------------------------------------
class AsymmetricLoss(nn.Module):
    """Asymmetric Loss for multi-label imbalance (Ridnik et al. 2021), optimized variant.

    gamma_neg=4, gamma_pos=0, clip=0.05 maximizes macro AUC under extreme sparsity.
    """

    def __init__(self, gamma_neg: float = 4.0, gamma_pos: float = 0.0, clip: float = 0.05,
                 eps: float = 1e-8, disable_tilde_gamma: bool = True):
        super().__init__()
        self.gamma_neg, self.gamma_pos, self.clip = gamma_neg, gamma_pos, clip
        self.eps = eps
        self.disable_tilde_gamma = disable_tilde_gamma

    def forward(self, logits: torch.Tensor, targets: torch.Tensor,
                sample_weight: Optional[torch.Tensor] = None) -> torch.Tensor:
        probs = torch.sigmoid(logits)
        p_pos = probs
        p_neg = 1.0 - probs
        if self.clip is not None and self.clip > 0:
            p_neg = (p_neg + self.clip).clamp(max=1.0)
        loss_pos = targets * torch.log(p_pos.clamp(min=self.eps))
        loss_neg = (1.0 - targets) * torch.log(p_neg.clamp(min=self.eps))
        loss = -(loss_pos + loss_neg)
        # Asymmetric focusing
        tilde_gamma_neg = self.gamma_neg if self.disable_tilde_gamma else 0.0
        tilde_gamma_pos = self.gamma_pos if self.disable_tilde_gamma else 0.0
        loss = loss * ((probs - tilde_gamma_neg).clamp(min=0) ** self.gamma_neg * (1 - targets) +
                       (1 - probs - tilde_gamma_pos).clamp(min=0) ** self.gamma_pos * targets)
        if sample_weight is not None:
            sw = sample_weight.view(-1, 1)
            loss = loss * sw
            denom = sw.expand_as(loss).sum().clamp(min=1.0)
            return loss.sum() / denom
        return loss.mean()


class MacroAUC:
    """Competition metric: mean over classes of ROC-AUC; crash-safe for degenerate batches."""

    def __init__(self, n_classes: int = 12):
        self.n = n_classes
        self.reset()

    def reset(self) -> None:
        self.y: List[np.ndarray] = []
        self.p: List[np.ndarray] = []

    def update(self, probs: torch.Tensor, targets: torch.Tensor) -> None:
        self.p.append(probs.detach().float().cpu().numpy())
        self.y.append(targets.detach().cpu().numpy())

    def compute(self) -> Tuple[float, Dict[str, float]]:
        if not self.y:
            return float("nan"), {}
        y = np.concatenate(self.y, axis=0)
        p = np.concatenate(self.p, axis=0)
        per_class: Dict[str, float] = {}
        aucs: List[float] = []
        try:
            from sklearn.metrics import roc_auc_score
            for j in range(self.n):
                col = y[:, j]
                if len(np.unique(col)) < 2:      # single-class column -> AUC undefined; skip gracefully
                    per_class[f"class_{j}"] = float("nan")
                    continue
                a = float(roc_auc_score(col, p[:, j]))
                per_class[f"class_{j}"] = a
                aucs.append(a)
        except ImportError:
            # Manual rank-based AUC fallback (Mann-Whitney U).
            for j in range(self.n):
                col = y[:, j]
                pos, neg = p[col == 1, j], p[col == 0, j]
                if pos.size == 0 or neg.size == 0:
                    continue
                ranks = np.argsort(np.argsort(np.concatenate([pos, neg]))) + 1
                r_pos = ranks[:pos.size].sum()
                a = (r_pos - pos.size * (pos.size + 1) / 2) / (pos.size * neg.size)
                per_class[f"class_{j}"] = float(a)
                aucs.append(float(a))
        macro = float(np.mean(aucs)) if aucs else float("nan")
        return macro, per_class


# --------------------------------------------------------------------------------------
# SECTION 7: TRAINING LOOP (AMP, EMA, warmup+cosine, GroupKFold)
# --------------------------------------------------------------------------------------
class EMA:
    def __init__(self, model: nn.Module, decay: float):
        self.decay = decay
        self.shadow = {k: v.detach().clone().float() for k, v in model.state_dict().items()
                       if v.dtype.is_floating_point}
        self.other = {k: v.detach().clone() for k, v in model.state_dict().items()
                      if not v.dtype.is_floating_point}

    @torch.no_grad()
    def update(self, model: nn.Module) -> None:
        for k, v in model.state_dict().items():
            if k in self.shadow:
                self.shadow[k].mul_(self.decay).add_(v.detach().float(), alpha=1 - self.decay)
            else:
                self.other[k] = v.detach().clone()

    def state_dict(self) -> Dict[str, torch.Tensor]:
        out = dict(self.shadow)
        out.update(self.other)
        return out

    def apply_to(self, model: nn.Module) -> None:
        model.load_state_dict(self.state_dict(), strict=False)


def get_lr(cfg: Config, base_lr: float, epoch_frac: float) -> float:
    """Linear warmup for WARMUP_EPOCHS then cosine annealing to 0."""
    if cfg.WARMUP_EPOCHS > 0 and epoch_frac < cfg.WARMUP_EPOCHS:
        return base_lr * (epoch_frac / max(1e-6, float(cfg.WARMUP_EPOCHS)))
    prog = (epoch_frac - cfg.WARMUP_EPOCHS) / max(1e-6, float(cfg.EPOCHS - cfg.WARMUP_EPOCHS))
    prog = min(1.0, max(0.0, prog))
    return base_lr * 0.5 * (1.0 + math.cos(math.pi * prog))


@torch.no_grad()
def predict(model: nn.Module, batch: Dict[str, torch.Tensor], device: str, amp: bool) -> torch.Tensor:
    batch = {k: (v.to(device, non_blocking=True) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
    with torch.autocast(device_type=device, dtype=torch.float16, enabled=amp and device == "cuda"):
        logits = model(batch)
    return torch.sigmoid(logits.float())


def run_epoch(model: nn.Module, loader: DataLoader, optimizer: Optional[torch.optim.Optimizer],
              criterion: AsymmetricLoss, device: str, cfg: Config,
              ema: Optional[EMA], lr_fn=None, epoch_idx: float = 0.0,
              metrics: Optional[MacroAUC] = None, log_every: int = 25) -> float:
    train = optimizer is not None
    model.train(train)
    if metrics is not None:
        metrics.reset()
    total_loss = 0.0
    n_batches = 0
    for step, batch in enumerate(loader):
        uids = batch.pop("uids")  # type: ignore[misc]
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        if train:
            assert optimizer is not None
            if lr_fn is not None:
                lr = lr_fn(epoch_idx + step / max(1, len(loader)))
                for g in optimizer.param_groups:
                    g["lr"] = lr
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=cfg.MIXED_PRECISION and device == "cuda"):
                logits = model(batch)
                loss = criterion(logits, batch["target"], batch["weight"])
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            if ema is not None:
                ema.update(model)
        else:
            assert metrics is not None
            with torch.autocast(device_type=device, dtype=torch.float16, enabled=cfg.MIXED_PRECISION and device == "cuda"):
                logits = model(batch)
            probs = torch.sigmoid(logits.float())
            metrics.update(probs, batch["target"])
        total_loss += float(loss.detach().item())
        n_batches += 1
        if train and (step + 1) % log_every == 0:
            cur_lr = optimizer.param_groups[0]["lr"] if optimizer else 0.0
            print(f"  [{mode_str(train)}] step {step+1}/{len(loader)} loss={total_loss/n_batches:.4f} lr={cur_lr:.2e}", flush=True)
    return total_loss / max(1, n_batches)


def mode_str(train: bool) -> str:
    return "train" if train else "valid"


def make_folds(df: pd.DataFrame, n_folds: int, seed: int) -> List[Tuple[np.ndarray, np.ndarray]]:
    """Stratified GroupKFold by StudyInstanceUID (multi-label stratification via label-hash bucketing)."""
    from sklearn.model_selection import StratifiedGroupKFold
    labels_bin = df[TARGET_COLS].fillna(0).astype(int)
    # Bucket stratification key: primary positive class (most frequent informative), fallback "ALLNEG".
    any_pos = labels_bin.sum(axis=1) > 0
    strat_key = np.where(any_pos, labels_bin.idxmax(axis=1), "ALLNEG")
    sgkf = StratifiedGroupKFold(n_splits=n_folds, shuffle=True, random_state=seed)
    folds = list(sgkf.split(df.index, y=strat_key, groups=df["StudyInstanceUID"].values))
    return folds


def train_pipeline(args: argparse.Namespace) -> None:
    cfg = Config()
    cfg.BACKBONE = args.backbone
    cfg.EPOCHS = args.epochs
    cfg.BATCH_SIZE = args.batch_size
    cfg.NUM_WORKERS = args.num_workers
    cfg.AGGREGATOR = args.aggregator
    set_determinism(cfg.SEED)
    device = cfg.DEVICE
    print(f"Config: {json.dumps(asdict(cfg), default=str)}")

    manifest = pd.read_csv(args.manifest)
    labels = pd.read_csv(args.labels)
    # Restrict to studies whose volumes actually exist.
    vol_dir = Path(args.volumes_root)
    existing = {str(Path(r).name).rsplit("_", 1)[0] for r in manifest.loc[manifest["status"].isin(["ok", "skipped"]), "VolumePath"]}
    labels = labels[labels["StudyInstanceUID"].astype(str).isin(existing)].reset_index(drop=True)
    if len(labels) == 0:
        raise RuntimeError("No labeled studies intersect with available preprocessed volumes.")

    folds = make_folds(labels, cfg.N_FOLDS, cfg.SEED)
    ckpt_dir = Path(args.ckpt_dir); ckpt_dir.mkdir(parents=True, exist_ok=True)

    fold_range = [args.fold] if args.fold >= 0 else list(range(cfg.N_FOLDS))
    oof_all: Dict[int, np.ndarray] = {}

    for fold in fold_range:
        tr_idx, va_idx = folds[fold]
        tr_df, va_df = labels.iloc[tr_idx], labels.iloc[va_idx]
        ds_tr = RSNAKneeDataset(tr_df, manifest, args.volumes_root, cfg, train=True)
        ds_va = RSNAKneeDataset(va_df, manifest, args.volumes_root, cfg, train=False)
        dl_tr = DataLoader(ds_tr, batch_size=cfg.BATCH_SIZE, shuffle=True, num_workers=cfg.NUM_WORKERS,
                           collate_fn=collate_knee, pin_memory=(device == "cuda"), drop_last=True)
        dl_va = DataLoader(ds_va, batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=cfg.NUM_WORKERS,
                           collate_fn=collate_knee, pin_memory=(device == "cuda"))

        model = RSNAMultiPlaneModel(cfg).to(device)
        ema = EMA(model, cfg.EMA_DECAY) if cfg.USE_EMA else None
        opt = torch.optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=cfg.WEIGHT_DECAY)
        crit = AsymmetricLoss(gamma_neg=4.0, gamma_pos=0.0, clip=0.05).to(device)
        metric = MacroAUC(len(cfg.TARGET_COLS))
        steps_per_epoch = max(1, len(dl_tr))
        total_steps = cfg.EPOCHS * steps_per_epoch
        start_step = [0]

        def lr_fn(frac_epochs: float) -> float:
            return get_lr(cfg, cfg.LR, frac_epochs)

        best_auc = -1.0
        hist: List[Dict[str, float]] = []
        for epoch in range(cfg.EPOCHS):
            t0 = time.time()
            tl = run_epoch(model, dl_tr, opt, crit, device, cfg, ema, lr_fn, epoch_idx=float(epoch))
            # Validate with EMA weights (swap-in, swap-out).
            raw_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            if ema is not None:
                ema.apply_to(model)
            model.eval()
            vl = run_epoch(model, dl_va, None, crit, device, cfg, None, None, metrics=metric)
            auc, per_class = metric.compute()
            if ema is not None:
                model.load_state_dict(raw_state)
            dt = time.time() - t0
            print(f"[fold {fold}] epoch {epoch+1}/{cfg.EPOCHS} train_loss={tl:.4f} val_loss={vl:.4f} "
                  f"val_macro_AUC={auc:.4f} ({dt:.1f}s)", flush=True)
            hist.append({"epoch": epoch + 1, "train_loss": tl, "val_loss": vl, "val_auc": auc})
            if np.isfinite(auc) and auc > best_auc:
                best_auc = auc
                payload = {
                    "model_state": model.state_dict(),
                    "ema_state": ema.state_dict() if ema is not None else None,
                    "optimizer_state": opt.state_dict(),
                    "config": asdict(cfg),
                    "fold": fold,
                    "best_auc": best_auc,
                    "backbone": cfg.BACKBONE,
                }
                torch.save(payload, ckpt_dir / f"best_auc_fold_{fold}.pt")
                print(f"  -> saved new best checkpoint (AUC={best_auc:.4f})", flush=True)
        with open(ckpt_dir / f"history_fold_{fold}_{Path(cfg.BACKBONE).name}.json", "w") as fh:
            json.dump({"history": hist, "best_auc": best_auc}, fh, indent=2)
        print(f"[fold {fold}] DONE best val macro AUC = {best_auc:.4f}", flush=True)

    # Fold summary
    summary_fp = ckpt_dir / "fold_summary.json"
    summaries = {}
    for f in fold_range:
        hb = sorted(ckpt_dir.glob(f"history_fold_{f}_*.json"))
        if hb:
            summaries[f] = json.loads(hb[-1].read_text()).get("best_auc")
    summary_fp.write_text(json.dumps(summaries, indent=2))
    print("Fold best-AUC summary:", summaries)


def main(argv: Optional[List[str]] = None) -> int:
    ap = argparse.ArgumentParser(description="RSNA Knee multi-plane 2.5D trainer")
    ap.add_argument("--manifest", required=True)
    ap.add_argument("--labels", required=True, help="train_pseudo_labels.csv from Module 2")
    ap.add_argument("--volumes-root", required=True)
    ap.add_argument("--ckpt-dir", default="/kaggle/working/checkpoints")
    ap.add_argument("--backbone", default="convnext_nano.in12k_ft_in1k")
    ap.add_argument("--aggregator", default="bigru", choices=["bigru", "mhsa"])
    ap.add_argument("--epochs", type=int, default=6)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--num-workers", type=int, default=4)
    ap.add_argument("--fold", type=int, default=-1, help="fold index or -1 for all folds")
    ns = ap.parse_args(argv)
    train_pipeline(ns)
    return 0


if __name__ == "__main__":
    pass  # driven from notebook cells


In [ ]:
# --- Stage 3a: train convnext_nano folds ---------------------------------------------------
import argparse as _ap
if os.path.exists(MANIFEST) and os.path.exists(PSEUDO_CSV) and os.path.isdir(VOL_ROOT):
    args = _ap.Namespace(manifest=MANIFEST, labels=PSEUDO_CSV, volumes_root=VOL_ROOT,
                         ckpt_dir=os.path.join(CKPT_ROOT, "convnext_nano"),
                         backbone="convnext_nano.in12k_ft_in1k", aggregator="bigru",
                         epochs=1 if SMOKE_TEST else 6, batch_size=8, num_workers=4,
                         fold=0 if SMOKE_TEST else -1)
    train_pipeline(args)
else:
    print("[Stage 3] prerequisites missing - skipped")

## Stage 4 · Module 4 source — `submission_notebook.py`

In [ ]:
"""
MODULE 4 — Kaggle-Compliant Offline Inference & Ensemble Submission (RSNA Knee).

Design goals
------------
1. STRICTLY OFFLINE: no pip installs, no HTTP calls; all weights come from attached datasets.
2. FAST DYNAMIC DICOM READER for test studies (no preprocessed .npy available at submit time):
   slice sorting -> percentile clip -> linspace resample to K=16 -> resize 256x256 -> tensor.
3. DEFENSIVE: any unreadable/corrupt study falls back to training-set prior probabilities.
4. TTA: original + horizontal-flip averaged.
5. ENSEMBLE: 5 fold checkpoints x {convnext_nano, tf_efficientnet_b0_ns} rank-averaged predictions.
6. VALIDATION: exact row count vs test.csv, exact header order, values in [0,1], non-null.

Paste this into the Kaggle "submission" notebook cells (Internet OFF, GPU ON, <=9h budget).
It is also importable as a module (`python submission_notebook.py --dry-run` for smoke tests).
"""

from __future__ import annotations

import argparse
import json
import os
import sys
import time
import traceback
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

# --------------------------------------------------------------------------------------
# Shared constants / config (must match Module 3 exactly)
# --------------------------------------------------------------------------------------
TARGET_COLS: List[str] = [
    "ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
    "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture",
]
SUBMISSION_HEADER: List[str] = ["StudyInstanceUID"] + TARGET_COLS


@dataclass
class InferConfig:
    SEED: int = 42
    IMAGE_SIZE: Tuple[int, int] = (256, 256)
    NUM_SLICES: int = 16
    BATCH_SIZE: int = 8                       # studies per batch during inference
    IN_CHANNELS: int = 3
    NUM_PLANES: int = 3
    FEATURE_DIM: int = 256
    AGGREGATOR: str = "bigru"
    ATTENTION_HEADS: int = 4
    DROPOUT: float = 0.25
    MIXED_PRECISION: bool = True
    BACKBONES: List[str] = field(default_factory=lambda: [
        "convnext_nano.in12k_ft_in1k", "tf_efficientnet_b0.ns_jft_in1k",
    ])
    N_FOLDS: int = 5
    TIME_BUDGET_HOURS: float = 7.5            # stop early with partial ensemble if exceeded
    DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"


np.random.seed(InferConfig.SEED)
torch.manual_seed(InferConfig.SEED)

# --------------------------------------------------------------------------------------
# Import model/dataset classes from train.py when available; otherwise define inline.
# The submission notebook must be self-contained if train.py isn't attached, so we
# re-export via try/except with full inline fallback definitions.
# --------------------------------------------------------------------------------------
_INLINE_DEFINITIONS = False
try:
    from train import (  # type: ignore
        Config as TrainConfig, RSNAMultiPlaneModel, RSNAKneeDataset, collate_knee, build_backbone,
    )
except Exception:
    _INLINE_DEFINITIONS = True

if _INLINE_DEFINITIONS:
    @dataclass
    class TrainConfig:
        SEED: int = 42
        IMAGE_SIZE: Tuple[int, int] = (256, 256)
        NUM_SLICES: int = 16
        BATCH_SIZE: int = 8
        NUM_WORKERS: int = 4
        BACKBONE: str = "convnext_nano.in12k_ft_in1k"
        IN_CHANNELS: int = 3
        NUM_PLANES: int = 3
        FEATURE_DIM: int = 256
        AGGREGATOR: str = "bigru"
        ATTENTION_HEADS: int = 4
        DROPOUT: float = 0.25
        LR: float = 3e-4
        WEIGHT_DECAY: float = 1e-2
        EPOCHS: int = 6
        WARMUP_EPOCHS: int = 1
        N_FOLDS: int = 5
        MIXED_PRECISION: bool = True
        USE_EMA: bool = True
        EMA_DECAY: float = 0.995
        PSEUDO_WEIGHT: float = 0.5
        TARGET_COLS: List[str] = field(default_factory=lambda: list(TARGET_COLS))
        DEVICE: str = "cuda" if torch.cuda.is_available() else "cpu"

    def build_backbone(name: str, in_chans: int) -> Tuple[nn.Module, int]:
        """Offline-safe timm backbone builder with local CNN fallback."""
        try:
            import timm
            errors: List[str] = []
            for cand in (name, name.split(".")[0]):
                try:
                    m = timm.create_model(cand, pretrained=False, in_chans=in_chans, num_classes=0)
                    dim = getattr(m, "num_features", None) or 512
                    return m, int(dim)
                except Exception as exc:
                    errors.append(str(exc))
            raise RuntimeError("; ".join(errors[:2]))
        except Exception:
            class FallbackCNN(nn.Module):
                def __init__(self, c: int, dim: int = 256):
                    super().__init__()
                    def block(ci, co):
                        return nn.Sequential(nn.Conv2d(ci, co, 3, stride=2, padding=1, bias=False),
                                             nn.BatchNorm2d(co), nn.SiLU(inplace=True))
                    self.net = nn.Sequential(block(c, 32), block(32, 64), block(64, 128),
                                             block(128, 256), nn.AdaptiveAvgPool2d(1), nn.Flatten())
                    self.num_features = dim
                def forward(self, x):
                    return self.net(x)
            return FallbackCNN(in_chans, 256), 256

    class SliceAggregator(nn.Module):
        def __init__(self, dim: int, mode: str = "bigru", heads: int = 4, dropout: float = 0.25):
            super().__init__()
            self.mode = mode
            if mode == "bigru":
                self.gru = nn.GRU(dim, dim // 2, num_layers=1, batch_first=True, bidirectional=True, dropout=dropout)
            else:
                self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=heads, dropout=dropout, batch_first=True)
                self.norm1 = nn.LayerNorm(dim)
            self.pool_query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
            self.pool_attn = nn.MultiheadAttention(embed_dim=dim, num_heads=max(1, heads // 2),
                                                   batch_first=True, dropout=dropout)
            self.norm = nn.LayerNorm(dim)
            self.drop = nn.Dropout(dropout)

        def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
            if self.mode == "bigru":
                h, _ = self.gru(x)
            else:
                att, _ = self.attn(x, x, x)
                h = self.norm1(x + att)
            q = self.pool_query.expand(x.shape[0], -1, -1)
            pooled, _ = self.pool_attn(q, h, h, need_weights=False)
            return self.drop(self.norm(pooled.squeeze(1)))

    class RSNAMultiPlaneModel(nn.Module):
        def __init__(self, cfg: TrainConfig):
            super().__init__()
            self.cfg = cfg
            backbone, feat_dim = build_backbone(cfg.BACKBONE, cfg.IN_CHANNELS)
            self.backbone = backbone
            self.proj = nn.Sequential(nn.Linear(feat_dim, cfg.FEATURE_DIM), nn.ReLU(inplace=True))
            self.aggregators = nn.ModuleList([
                SliceAggregator(cfg.FEATURE_DIM, mode=cfg.AGGREGATOR, heads=cfg.ATTENTION_HEADS, dropout=cfg.DROPOUT)
                for _ in range(cfg.NUM_PLANES)])
            self.meta_embed = nn.Embedding(4, 16)
            self.flags_proj = nn.Linear(3 * 2, 32)
            self.plane_embed = nn.Embedding(cfg.NUM_PLANES, 8)
            fused_dim = cfg.FEATURE_DIM * cfg.NUM_PLANES + 32 + cfg.NUM_PLANES * 8
            self.head = nn.Sequential(
                nn.Dropout(cfg.DROPOUT), nn.Linear(fused_dim, 512), nn.GELU(),
                nn.Dropout(cfg.DROPOUT), nn.Linear(512, len(cfg.TARGET_COLS)))

        def forward(self, batch: Dict[str, torch.Tensor]) -> torch.Tensor:
            vols = batch["volumes"]
            b, p, d, c, h, w = vols.shape
            x = vols.reshape(b * p * d, c, h, w)
            feats = self.backbone(x)
            feats = self.proj(feats.float())
            feats = feats.reshape(b * p, d, -1)
            plane_masks = batch["plane_mask"].reshape(b * p)
            outs = []
            for pi in range(p):
                sel = slice(pi, b * p, p)
                fp, mp = feats[sel], plane_masks[sel]
                agg = self.aggregators[pi](fp, mask=None)
                outs.append(agg * mp.unsqueeze(-1))
            plane_feats = torch.cat(outs, dim=-1)
            flags = torch.cat([batch["fluid_flags"], batch["fat_flags"]], dim=-1)
            flag_emb = self.flags_proj(flags)
            plane_ids = torch.arange(p, device=vols.device).unsqueeze(0).expand(b, -1)
            p_emb = self.plane_embed(plane_ids).reshape(b, -1)
            fused = torch.cat([plane_feats, flag_emb, p_emb], dim=-1)
            return self.head(fused)

    def collate_knee(batch: List[Dict[str, torch.Tensor]]) -> Dict[str, torch.Tensor]:
        uids = [b.pop("uid") for b in batch]
        out = {k: torch.stack([b[k] for b in batch], dim=0) for k in
               ["volumes", "plane_mask", "fluid_flags", "fat_flags", "meta", "target", "weight"]}
        out["uids"] = uids  # type: ignore[assignment]
        return out


# --------------------------------------------------------------------------------------
# Fast dynamic DICOM reader (test-time, on-the-fly)
# --------------------------------------------------------------------------------------
def _safe_get(ds, name, default=None):
    try:
        v = getattr(ds, name, default)
    except Exception:
        return default
    return default if v is None else v


def read_series_onthefly(series_dir: Path, num_slices: int, image_size: Tuple[int, int],
                         p_low: float = 0.5, p_high: float = 99.5) -> Tuple[np.ndarray, Dict[str, float]]:
    """Decode one series dir -> (uint8 volume (K,H,W), metadata dict {fluid, fat}).

    Raises RuntimeError on total decode failure (caller imputes priors).
    """
    import pydicom
    files = [Path(e.path) for e in os.scandir(str(series_dir)) if e.is_file()]
    if not files:
        raise RuntimeError(f"empty series {series_dir}")
    dss = []
    n_err = 0
    for fp in files:
        try:
            ds = pydicom.dcmread(str(fp), force=True)
            if "PixelData" in ds or hasattr(ds, "pixel_data_array"):
                dss.append(ds)
            else:
                n_err += 1
        except Exception:
            n_err += 1
    if not dss:
        raise RuntimeError(f"no readable slices in {series_dir} ({n_err} errors)")

    def zkey(ds):
        ipp = _safe_get(ds, "ImagePositionPatient", None)
        try:
            return float(ipp[2])
        except Exception:
            pass
        try:
            return float(_safe_get(ds, "SliceLocation", float(_safe_get(ds, "InstanceNumber", 0))))
        except Exception:
            return 0.0
    try:
        dss = sorted(dss, key=zkey)
    except Exception:
        pass

    planes = str(_safe_get(dss[0], "AnatomicalPlaneType", "") or "").upper()
    desc_blob = (str(_safe_get(dss[0], "SeriesDescription", "") or "") + " " +
                 str(_safe_get(dss[0], "SequenceName", "") or "")).upper()
    if not planes or planes == "OTHER":
        iop = _safe_get(dss[0], "ImageOrientationPatient", None)
        try:
            row = np.array([float(x) for x in iop[:3]]); col = np.array([float(x) for x in iop[3:]])
            n = np.cross(row, col); ax = int(np.argmax(np.abs(n)))
            planes = {0: "AXIAL", 1: "CORONAL", 2: "SAGITTAL"}.get(ax, "UNKNOWN")
            # normal direction convention varies; fall back to description keywords
            if "SAG" in desc_blob: planes = "SAGITTAL"
            elif "COR" in desc_blob: planes = "CORONAL"
            elif "AXI" in desc_blob or "TRA" in desc_blob: planes = "AXIAL"
        except Exception:
            planes = "UNKNOWN"
    fluid = float(any(t in desc_blob for t in ("PDFS", "T2 FS", "T2FS", "STIR", "FLAIR", "FS", "FATSAT", "SPIR")))
    fat = float(any(t in desc_blob for t in ("FS", "FATSAT", "SPIR", "STIR")))

    arrs: List[np.ndarray] = []
    decode_fail = 0
    for ds in dss:
        try:
            a = ds.pixel_array
            if a.ndim == 3:
                a = a.mean(axis=0) if a.shape[0] <= 16 else a.reshape(a.shape[0], -1).mean(axis=1).reshape(a.shape[1:])
            a = a.astype(np.float32)
            slope = _safe_get(ds, "RescaleSlope", 1.0); inter = _safe_get(ds, "RescaleIntercept", 0.0)
            try:
                a = a * float(slope) + float(inter)
            except Exception:
                pass
            arrs.append(a)
        except Exception:
            decode_fail += 1
    if decode_fail > max(1, len(dss) // 2) or not arrs:
        raise RuntimeError(f"{decode_fail}/{len(dss)} undecodable slices in {series_dir} (missing pylibjpeg decoders?)")
    while len(arrs) < len(dss):
        arrs.append(np.zeros_like(arrs[0]))

    vol = np.stack(arrs, axis=0).astype(np.float32)
    ref = arrs[0]
    finite = vol[np.isfinite(vol)]
    lo = float(np.percentile(finite, p_low)) if finite.size else 0.0
    hi = float(np.percentile(finite, p_high)) if finite.size else 1.0
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = -1.0, 1.0
    vol = np.nan_to_num((vol - lo) / (hi - lo), nan=0.0, posinf=1.0, neginf=0.0).clip(0, 1)
    vol = (vol * 255.0).round().astype(np.uint8)

    # depth resample to K
    d = vol.shape[0]
    if d != num_slices:
        idx = np.linspace(0, d - 1, num_slices).round().astype(int) if d > 1 else np.zeros(num_slices, dtype=int)
        vol = vol[idx]
    # spatial resize
    if vol.shape[1:] != tuple(image_size):
        try:
            import cv2
            vol = np.stack([cv2.resize(v, (image_size[1], image_size[0]), interpolation=cv2.INTER_AREA) for v in vol])
        except Exception:
            from PIL import Image
            vol = np.stack([np.asarray(Image.fromarray(v).resize((image_size[1], image_size[0]))) for v in vol])
    meta = {"plane": planes, "fluid": fluid, "fat": fat}
    assert vol.shape[0] == num_slices and vol.shape[1] == image_size[0] and vol.shape[2] == image_size[1]
    return vol, meta


def load_study_volumes(study_series_dir: Path, cfg: InferConfig) -> Dict[str, object]:
    """Assemble the 3-plane tensor bundle for one study with defensive zero-padding."""
    slots: Dict[str, Tuple[Optional[np.ndarray], Dict[str, float]]] = {
        "SAGITTAL": (None, {}), "CORONAL": (None, {}), "AXIAL": (None, {})}
    order = ["SAGITTAL", "CORONAL", "AXIAL"]
    extras: List[Tuple[np.ndarray, Dict[str, float]]] = []
    entries = [e for e in os.scandir(str(study_series_dir)) if e.is_dir()]
    for e in entries:
        try:
            vol, meta = read_series_onthefly(Path(e.path), cfg.NUM_SLICES, cfg.IMAGE_SIZE)
        except Exception:
            continue
        pl = str(meta.get("plane", "UNKNOWN")).upper()
        placed = False
        for key in order:
            if key in pl and slots[key][0] is None:
                slots[key] = (vol, meta); placed = True; break
        if not placed:
            extras.append((vol, meta))
    # Fill empty slots with leftover series (keeps model fed even if plane tags are bad).
    for key in order:
        if slots[key][0] is None and extras:
            slots[key] = extras.pop(0)

    vols, masks, fluids, fats = [], [], [], []
    for key in order:
        v, m = slots[key]
        if v is None:
            v = np.zeros((cfg.NUM_SLICES, *cfg.IMAGE_SIZE), dtype=np.uint8)
            m = {"fluid": 0.0, "fat": 0.0}
            masks.append(0.0)
        else:
            masks.append(1.0)
        vols.append(v)
        fluids.append(float(m.get("fluid", 0.0)))
        fats.append(float(m.get("fat", 0.0)))
    stack = np.stack(vols, axis=0).astype(np.float32) / 255.0
    stack = np.repeat(stack[:, :, None], cfg.IN_CHANNELS, axis=2)     # (P,D,C,H,W)
    any_readable = max(masks) > 0.0
    return {"volumes": stack, "plane_mask": np.array(masks, np.float32),
            "fluid_flags": np.array(fluids, np.float32), "fat_flags": np.array(fats, np.float32),
            "meta": np.array([max(fluids), max(fats), float(sum(masks)) / 3.0, masks[2]], np.float32),
            "readable": bool(any_readable)}


# --------------------------------------------------------------------------------------
# Checkpoint loading & ensemble inference
# --------------------------------------------------------------------------------------
def find_checkpoints(weights_root: str | Path, backbones: List[str], n_folds: int) -> List[Path]:
    root = Path(weights_root)
    found: List[Path] = []
    for bb in backbones:
        stem = bb.split(".")[0].replace("-", "_")
        for f in range(n_folds):
            cands = list(root.rglob(f"best_auc_fold_{f}.pt")) + \
                    list(root.rglob(f"*{stem}*fold_{f}*.pt")) + \
                    list(root.rglob(f"*fold_{f}*{stem}*.pt"))
            uniq = {c.resolve(): c for c in cands}
            for c in uniq.values():
                if c not in found:
                    found.append(c)
                    break
    return found


def make_train_cfg_from_ckpt(payload: Dict[str, object], fallback_backbone: str) -> TrainConfig:
    cfg_d = payload.get("config", {}) if isinstance(payload, dict) else {}
    tc = TrainConfig()
    for k, v in (cfg_d or {}).items():
        if hasattr(tc, k) and k not in ("DEVICE", "TARGET_COLS"):
            setattr(tc, k, v)
    tc.BACKBONE = str(payload.get("backbone", fallback_backbone)) if isinstance(payload, dict) else fallback_backbone
    return tc


@torch.inference_mode()
def predict_one_model(model: nn.Module, loader_iter, device: str, amp: bool,
                      flip: bool = False) -> Tuple[np.ndarray, np.ndarray]:
    """Run inference over pre-assembled study tensors; optionally horizontally flip volumes."""
    model.eval()
    probs_all: List[np.ndarray] = []
    uids_all: List[str] = []
    for uids, bundle in loader_iter:
        vols = bundle["volumes"]
        if flip:
            vols = vols[..., ::-1]
        x = {
            "volumes": torch.from_numpy(vols).to(device, non_blocking=True),
            "plane_mask": torch.from_numpy(bundle["plane_mask"]).to(device),
            "fluid_flags": torch.from_numpy(bundle["fluid_flags"]).to(device),
            "fat_flags": torch.from_numpy(bundle["fat_flags"]).to(device),
            "meta": torch.from_numpy(bundle["meta"]).to(device),
            "target": torch.zeros((len(uids), len(TARGET_COLS))),
            "weight": torch.ones(len(uids)),
        }
        with torch.autocast(device_type=device, dtype=torch.float16, enabled=amp and device == "cuda"):
            logits = model(x)
        probs_all.append(torch.sigmoid(logits.float()).cpu().numpy())
        uids_all.extend(uids)
    return np.concatenate(probs_all, axis=0) if probs_all else np.zeros((0, len(TARGET_COLS))), np.array(uids_all)


def build_study_batches(studies: List[str], series_root: Path, cfg: InferConfig,
                        chunk: int = 48) -> List[Tuple[List[str], Dict[str, np.ndarray]]]:
    """Group raw DICOM studies into inference batches; failed reads flagged for imputation."""
    batches: List[Tuple[List[str], Dict[str, np.ndarray]]] = []
    buf_u: List[str] = []
    buf_v: List[np.ndarray] = []
    buf_pm: List[np.ndarray] = []
    buf_fl: List[np.ndarray] = []
    buf_fa: List[np.ndarray] = []
    buf_me: List[np.ndarray] = []
    for uid in studies:
        sdir = series_root / uid
        ok = False
        try:
            if sdir.is_dir():
                bundle = load_study_volumes(sdir, cfg)
                ok = bundle["readable"]
                if ok:
                    buf_v.append(bundle["volumes"]); buf_pm.append(bundle["plane_mask"])
                    buf_fl.append(bundle["fluid_flags"]); buf_fa.append(bundle["fat_flags"])
                    buf_me.append(bundle["meta"]); buf_u.append(uid)
        except Exception:
            ok = False
        if not ok:
            buf_u.append(uid)
            buf_v.append(np.zeros((cfg.NUM_PLANES, cfg.NUM_SLICES, cfg.IN_CHANNELS, *cfg.IMAGE_SIZE), np.float32))
            buf_pm.append(np.zeros(cfg.NUM_PLANES, np.float32))
            buf_fl.append(np.zeros(cfg.NUM_PLANES, np.float32))
            buf_fa.append(np.zeros(cfg.NUM_PLANES, np.float32))
            buf_me.append(np.zeros(4, np.float32))
        if len(buf_u) >= chunk:
            batches.append((buf_u, {"volumes": np.stack(buf_v), "plane_mask": np.stack(buf_pm),
                                    "fluid_flags": np.stack(buf_fl), "fat_flags": np.stack(buf_fa),
                                    "meta": np.stack(buf_me)}))
            buf_u, buf_v, buf_pm, buf_fl, buf_fa, buf_me = [], [], [], [], [], []
    if buf_u:
        batches.append((buf_u, {"volumes": np.stack(buf_v), "plane_mask": np.stack(buf_pm),
                                "fluid_flags": np.stack(buf_fl), "fat_flags": np.stack(buf_fa),
                                "meta": np.stack(buf_me)}))
    return batches


def rank_average(pred_stack: np.ndarray) -> np.ndarray:
    """Per-model, per-column rank normalization followed by averaging.

    Rank-averaging removes calibration drift between fold/backbone models while the
    probability-mean (applied separately and blended by the caller) preserves magnitude.
    pred_stack: (M, N, C) -> (N, C) in [0, 1].
    """
    m, n, c = pred_stack.shape
    if m == 1:
        return pred_stack[0]
    out = np.empty_like(pred_stack, dtype=np.float64)
    for mi in range(m):
        for cj in range(c):
            col = pred_stack[mi, :, cj]
            order = np.argsort(col, kind="stable")
            ranks = np.empty(n, dtype=np.float64)
            ranks[order] = np.arange(n, dtype=np.float64)
            out[mi, :, cj] = ranks / max(1.0, float(n - 1))
    return out.mean(axis=0)


def compute_training_priors(labels_csv: Optional[str], manifest_status_csv: Optional[str]) -> np.ndarray:
    """Fallback prior probabilities per target used to impute unreadable test studies."""
    if labels_csv and Path(labels_csv).exists():
        df = pd.read_csv(labels_csv)
        cols = [c for c in TARGET_COLS if c in df.columns]
        if cols:
            priors = df[cols].fillna(0).mean(axis=0).values.astype(np.float64)
            full = np.zeros(len(TARGET_COLS), dtype=np.float64)
            for i, c in enumerate(cols):
                full[TARGET_COLS.index(c)] = priors[i]
            return np.clip(full, 0.01, 0.99)
    return np.full(len(TARGET_COLS), 0.10, dtype=np.float64)  # generic knee-abnormality base rate


def run_submission(
    test_csv: str,
    test_series_root: str,
    weights_root: str,
    out_csv: str,
    labels_for_priors: Optional[str] = None,
    dry_run: bool = False,
    max_studies: Optional[int] = None,
) -> pd.DataFrame:
    cfg = InferConfig()
    t_start = time.time()
    deadline = t_start + cfg.TIME_BUDGET_HOURS * 3600.0

    test_df = pd.read_csv(test_csv)
    if "StudyInstanceUID" not in test_df.columns:
        raise ValueError("test.csv missing StudyInstanceUID column")
    study_ids = test_df["StudyInstanceUID"].astype(str).tolist()
    submission_mode = len(test_df) > 3 or Path(test_series_root).exists()
    print(f"[INFO] rows={len(test_df)} | submission_mode={submission_mode} | device={cfg.DEVICE}")
    if max_studies:
        study_ids = study_ids[:max_studies]

    priors = compute_training_priors(labels_for_priors, None)
    series_root = Path(test_series_root)

    # ---- assemble inference batches (CPU-bound DICOM decode happens once) ----------
    print("[INFO] Decoding test DICOM series (on-the-fly)...", flush=True)
    if dry_run:
        batches = []
    else:
        batches = build_study_batches(study_ids, series_root, cfg, chunk=48)
    print(f"[INFO] Built {len(batches)} study batches.", flush=True)

    ckpts = find_checkpoints(weights_root, cfg.BACKBONES, cfg.N_FOLDS)
    if not ckpts:
        print("[WARN] No checkpoints found under", weights_root, "— submitting calibrated priors.")
        preds = np.tile(priors, (len(study_ids), 1))
        uids = np.array(study_ids)
    else:
        print(f"[INFO] Found {len(ckpts)} checkpoints: {[c.name for c in ckpts[:6]]}...")
        model_preds: List[np.ndarray] = []
        uid_order_ref: Optional[np.ndarray] = None
        for ci, ck in enumerate(ckpts):
            if time.time() > deadline:
                print("[WARN] Time budget reached; finalizing with partial ensemble.")
                break
            try:
                payload = torch.load(ck, map_location="cpu", weights_only=False)
            except TypeError:
                payload = torch.load(ck, map_location="cpu")
            tc = make_train_cfg_from_ckpt(payload, cfg.BACKBONES[ci % len(cfg.BACKBONES)])
            model = RSNAMultiPlaneModel(tc)
            state = payload.get("ema_state") or payload.get("model_state")
            missing, unexpected = model.load_state_dict(state, strict=False)
            if missing:
                print(f"[WARN] {ck.name}: missing keys {list(missing)[:4]} ...")
            model.to(cfg.DEVICE).eval()

            # Preload batches to tensors lazily per model to bound RAM.
            iters_orig = [(u, {k: v for k, v in b.items()}) for (u, b) in batches]
            probs_o, uids_o = predict_one_model(model, iters_orig, cfg.DEVICE, cfg.MIXED_PRECISION, flip=False)
            probs_f, _ = predict_one_model(model, iters_orig, cfg.DEVICE, cfg.MIXED_PRECISION, flip=True)
            probs_tta = (probs_o + probs_f) / 2.0
            del model
            if cfg.DEVICE == "cuda":
                torch.cuda.empty_cache()

            # Map back to the canonical study order; impute training priors for any missing rows.
            ordered = np.full((len(study_ids), len(TARGET_COLS)), np.nan)
            pos = {u: i for i, u in enumerate(study_ids)}
            for i, u in enumerate(uids_o):
                if u in pos:
                    ordered[pos[u]] = probs_tta[i]
            n_missing = int(np.isnan(ordered).any(axis=1).sum())
            if n_missing:
                ordered[np.isnan(ordered).any(axis=1)] = priors
                print(f"[INFO] Imputed training priors for {n_missing} unreadable studies.")
            model_preds.append(ordered)
            print(f"[INFO] Model {ci+1}/{len(ckpts)} ({Path(ck).parent.name}/{ck.name}) done "
                  f"in {(time.time()-t_start)/60:.1f} min elapsed", flush=True)

        if model_preds:
            stacked = np.stack(model_preds, axis=0)
            preds_rank = rank_average(stacked)
            preds_mean = stacked.mean(axis=0)
            # Blend rank-average with probability-mean (rank fixes calibration drift across folds).
            preds = np.clip(0.5 * preds_rank + 0.5 * preds_mean, 0.0, 1.0)
        else:
            preds = np.tile(priors, (len(study_ids), 1))

    sub = pd.DataFrame(preds, columns=TARGET_COLS)
    sub.insert(0, "StudyInstanceUID", study_ids)

    # ----------------------- SECTION 8 sanity checks --------------------------------
    expected_header = SUBMISSION_HEADER
    assert list(sub.columns) == expected_header, f"Header mismatch: {list(sub.columns)}"
    assert len(sub) == len(test_df), f"Row count mismatch: {len(sub)} vs {len(test_df)}"
    assert sub.notna().all().all(), "NaNs detected in submission"
    assert ((sub[TARGET_COLS] >= 0.0) & (sub[TARGET_COLS] <= 1.0)).all().all(), "Probabilities outside [0,1]"

    out_path = Path(out_csv)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    sub.to_csv(out_path, index=False)
    mins = (time.time() - t_start) / 60.0
    print(f"[OK] Wrote {out_path} | shape={sub.shape} | mean prob={sub[TARGET_COLS].values.mean():.4f} | elapsed={mins:.1f} min")
    return sub


def main(argv: Optional[List[str]] = None) -> int:
    ap = argparse.ArgumentParser(description="RSNA Knee offline ensemble submission pipeline")
    ap.add_argument("--test-csv", default="/kaggle/input/rsna-str-pkc-dataset/test.csv")
    ap.add_argument("--test-series-root", default="/kaggle/input/rsna-str-pkc-dataset/test_series")
    ap.add_argument("--weights-root", default="/kaggle/input/rsna-knee-weights")
    ap.add_argument("--labels-for-priors", default=None)
    ap.add_argument("--out-csv", default="submission.csv")
    ap.add_argument("--dry-run", action="store_true", help="Skip DICOM/model work; emit priors-only submission")
    ap.add_argument("--max-studies", type=int, default=None)
    ns = ap.parse_args(argv)
    run_submission(ns.test_csv, ns.test_series_root, ns.weights_root, ns.out_csv,
                   labels_for_priors=ns.labels_for_priors, dry_run=ns.dry_run, max_studies=ns.max_studies)
    return 0


if __name__ == "__main__":
    pass  # driven from notebook cells


In [ ]:
# --- Stage 4: offline ensemble inference & submission -------------------------------------
if os.path.exists(TEST_CSV):
    sub = run_submission(test_csv=TEST_CSV, test_series_root=TEST_SERIES,
                         weights_root=CKPT_ROOT, out_csv=os.path.join(WORK, "submission.csv"),
                         labels_for_priors=PSEUDO_CSV,
                         max_studies=16 if SMOKE_TEST else None)
    display(sub.head())
else:
    print("[Stage 4] test.csv missing - skipped")

## Stage 3b — second backbone for the ensemble (`tf_efficientnet_b0_ns`)

In [ ]:
# --- Stage 3b: efficientnet variant (drop the previous cell's re-execution of module source) --
if os.path.exists(MANIFEST) and os.path.exists(PSEUDO_CSV) and os.path.isdir(VOL_ROOT):
    import argparse as _ap2
    args = _ap2.Namespace(manifest=MANIFEST, labels=PSEUDO_CSV, volumes_root=VOL_ROOT,
                          ckpt_dir=os.path.join(CKPT_ROOT, "efficientnet_b0"),
                          backbone="tf_efficientnet_b0.ns_jft_in1k", aggregator="mhsa",
                          epochs=1 if SMOKE_TEST else 6, batch_size=8, num_workers=4,
                          fold=0 if SMOKE_TEST else -1)
    train_pipeline(args)
else:
    print("[Stage 3b] prerequisites missing - skipped")

## Post-run checklist
- [ ] Upload `preprocessed/` → private dataset `rsna-knee-preprocessed-256`
- [ ] Upload `checkpoints/*/*.pt` → private dataset `rsna-knee-weights`
- [ ] For the official submission: create a fresh notebook with **only Notebook 04**,
      attach competition data + weights dataset, **Internet OFF**, Submit.